# 00_ 01匯入資料並02整理成一張dataframe (請直接執行)

## 01_import dataset

### 01-1_import data

In [24]:
import os
import json
import pandas as pd

#https://drive.google.com/drive/folders/18qV82fNY3IIWu3BRoGqm_LNgJzE8Akbr?usp=drive_link
#base_dir = "/Users/Andypon/10_交大研究所/1141_01_機器學習與金融科技/data"
base_dir= '/Users/andyw.p.chen/Documents/Project/datasets'
#base_dir=  "c:\Users\user\Downloads\datasets"

def load_json_to_df(filename: str) -> pd.DataFrame:
    file_path = os.path.join(base_dir, filename)
    with open(file_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    # 如果是 { "target": {id: value, ...} }
    if isinstance(data, dict) and len(data) == 1 and isinstance(next(iter(data.values())), dict):
        key, inner = next(iter(data.items()))
        return pd.DataFrame(list(inner.items()), columns=["id", key])

    # dict of scalar
    if isinstance(data, dict):
        return pd.DataFrame([{"code": k, "desc": v} for k, v in data.items()])

    # list of dict
    elif isinstance(data, list):
        return pd.DataFrame(data)

    else:
        raise ValueError(f"Unsupported JSON structure in {filename}: {type(data)}")


def load_csv_to_df(filename: str) -> pd.DataFrame:
    """讀取 CSV 並轉為 DataFrame。"""
    return pd.read_csv(os.path.join(base_dir, filename))

# JSON 資料
##mcc_codes_df = load_json_to_df("mcc_codes.json")
train_fraud_labels_df = load_json_to_df("train_fraud_labels.json")

# CSV 資料
cards_df = load_csv_to_df("cards_data.csv")
transactions_df = load_csv_to_df("transactions_data.csv")
users_df = load_csv_to_df("users_data.csv")

# 簡單檢查
#print(mcc_codes_df.head())
#print(train_fraud_labels_df.head())
#print(cards_df.head())
#print(transactions_df.head())
#print(users_df.apthead())


### 01-2_rename variable in each data set

In [25]:
train_fraud_labels_df = train_fraud_labels_df.rename(columns={'id': 'transactions_id'})
train_fraud_labels_df = train_fraud_labels_df.rename(columns={'target': 'is_fraud'})

cards_df = cards_df.rename(columns={'id':'card_id'})

users_df = users_df.rename(columns={'id':'client_id'})

transactions_df = transactions_df.rename(columns={'mcc': 'mcc_code'})
transactions_df = transactions_df.rename(columns={'id': 'transaction_id'})




### 01-3_變數型態統一及缺失值處理

In [26]:
def add_missing_flags(df: pd.DataFrame, cols: list) -> pd.DataFrame:
    """
    在 DataFrame 中對指定欄位建立 missing flag 欄位
    flag=1 表示缺失值，flag=0 表示非缺失值
    
    參數
    ----
    df : pd.DataFrame
        輸入的資料框
    cols : list
        要檢查的欄位名稱清單
    
    回傳
    ----
    pd.DataFrame : 新的資料框 (含新增的 flag 欄位)
    """
    for col in cols:
        df[f"{col}_missing_flag"] = df[col].isna().astype(int)
    return df

transactions_df = add_missing_flags(transactions_df, ["merchant_state", "zip", "errors"])

/var/folders/yb/xnfk9z6x34527z3924bcjqcr0000gn/T/ipykernel_27178/743340693.py:18: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  df[f"{col}_missing_flag"] = df[col].isna().astype(int)
/var/folders/yb/xnfk9z6x34527z3924bcjqcr0000gn/T/ipykernel

In [27]:
##train_fraud_labels_df##
train_fraud_labels_df["is_fraud"]=train_fraud_labels_df["is_fraud"].astype("category") 
train_fraud_labels_df["transactions_id"]=train_fraud_labels_df["transactions_id"].astype(int) #合併資料需要

##cards_df##
cards_df["card_brand"]=cards_df["card_brand"].astype("category") 
cards_df["card_type"]=cards_df["card_type"].astype("category")
#####不要load這行 cards_df["expires"]=pd.to_datetime(cards_df["expires"], format="%m/%Y")
cards_df["expires"] = pd.to_datetime(cards_df["expires"], format="%m/%Y").dt.to_period("M")
cards_df["has_chip"]=cards_df["has_chip"].astype("category")

cards_df['credit_limit'] = cards_df['credit_limit'].replace(r'[\$,]', '', regex=True).astype(int)
#####不要load這行 cards_df["acct_open_date"]=pd.to_datetime(cards_df["acct_open_date"], format="%m/%Y")
cards_df["acct_open_date"] = pd.to_datetime(cards_df["acct_open_date"], format="%m/%Y").dt.to_period("M")
#####不要load這行 cards_df["year_pin_last_changed"]=pd.to_datetime(cards_df["year_pin_last_changed"], format="%Y")
cards_df["year_pin_last_changed"] = pd.to_datetime(cards_df["year_pin_last_changed"], format="%Y").dt.to_period("Y")
cards_df["card_on_dark_web"]=cards_df["card_on_dark_web"].astype("category") 

##users_df##
users_df["birth_year"] = pd.to_datetime(users_df["birth_year"], format="%Y").dt.to_period("Y")
users_df["birth_month"] = pd.to_datetime(users_df["birth_month"], format="%m").dt.to_period("M")
users_df["gender"]=users_df["gender"].astype("category") 
users_df['per_capita_income'] = users_df['per_capita_income'].replace(r'[\$,]', '', regex=True).astype(int)
users_df['yearly_income'] = users_df['yearly_income'].replace(r'[\$,]', '', regex=True).astype(int)
users_df['total_debt'] = users_df['total_debt'].replace(r'[\$,]', '', regex=True).astype(int)

##transactions_df##
transactions_df["date"] = pd.to_datetime(transactions_df["date"])
#浮點數轉整數原因確定？
transactions_df['amount'] = transactions_df['amount'].replace(r'[\$,]', '', regex=True).astype(float).astype(int)
##負數取log調成1
#transactions_df['amount'] = transactions_df['amount'].replace(r'[\$,]', '', regex=True).astype(float)

transactions_df["use_chip"]=transactions_df["use_chip"].astype("category") 

transactions_df.loc[
    transactions_df['merchant_city'].str.lower() == 'online',
    'merchant_state'
] = 'online'

transactions_df.loc[
    transactions_df['merchant_city'].str.lower() == 'online',
    'zip'
] = 20000 #原本是-1
## 我沒有全部改，這樣完之後仍有89006筆Missing，剩下都是在國外
transactions_df['zip'] = transactions_df['zip'].fillna(10000) #原本是-999
transactions_df["zip"]=transactions_df["zip"].astype("int64")

transactions_df['errors'] = transactions_df['errors'].astype('category')
transactions_df['errors'] = transactions_df['errors'].cat.add_categories('No_error').fillna('No_error')



/var/folders/yb/xnfk9z6x34527z3924bcjqcr0000gn/T/ipykernel_27178/1055933443.py:2: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  train_fraud_labels_df["is_fraud"]=train_fraud_labels_df["is_fraud"].astype("category")
/var/folders/yb/xnfk9z6x34

In [28]:
#cars one hot encoding
##統一類別變數轉dummy variable(要注意共線性問題，應刪掉其中之一)

#card_type 原始種類：Debit_57%, Credit_33%, Debit(Prepaid)_9%
#card_brand 原始種類：MasterCard_52%, Visa_38%, Amex_7%, Discovery_3%
#has_chip 原始種類：Yes_89%, No_11%
#card_on_dark_web 原始種類：No_0%
cols_to_encode = ['card_type', 'card_brand', 'has_chip']
cards_df[cols_to_encode] = cards_df[cols_to_encode].astype('category')
dummies_cards = pd.get_dummies(
    cards_df[cols_to_encode], 
    prefix=cols_to_encode, 
    dtype='uint8'
    )
cards_df = pd.concat([cards_df, dummies_cards], axis=1)

#use_chip 原始種類：Swiped_52%, Chipe_36%, Online_12%
dummies_use = pd.get_dummies(transactions_df['use_chip'], prefix='use_chip', dtype='uint8')
transactions_df = pd.concat([transactions_df, dummies_use], axis=1)

#gender 原始種類：Female_51%, Male_49%
dummies_gender = pd.get_dummies(users_df['gender'], prefix='gender', dtype='uint8')
users_df = pd.concat([users_df, dummies_gender], axis=1)


cards_df.drop(columns=["has_chip_NO","has_chip"], inplace=True)
transactions_df.drop(columns=["use_chip"], inplace=True)
users_df.drop(columns=["gender_Female"], inplace=True)

## 02_資料整併成一張dataframe

### 02-1_資料整併

In [29]:
#transactions_df.loc[transactions_df["transaction_id"] == 10649266] #transaction_id vs id

#原始資料筆數：13305915
### transactions_df+train_fraud_labels_df      left 會有4390952 missing values
merged = pd.merge(transactions_df, train_fraud_labels_df, left_on="transaction_id", right_on="transactions_id", how="outer")
### transactions_df train_fraud_labels_df(8914963) + users_df 對過去不會有missing values
merged = pd.merge(merged,users_df , left_on="client_id", right_on="client_id", how="left")
### transactions_df train_fraud_labels_df users_df + cards_df 對過去不會有missing values
merged = pd.merge(merged,cards_df , left_on="card_id", right_on="card_id", how="left")

#刪掉重複的columns
merged.drop(columns=["transactions_id"], inplace=True)
merged.drop(columns=["client_id_y"], inplace=True)

## 合併完之後最後處理is_fraud(原會有missing values問題)
merged["is_fraud"] = merged["is_fraud"].astype(str)
merged.loc[merged['is_fraud'].str.lower() == 'no','is_fraud'] = '0'
merged.loc[merged['is_fraud'].str.lower() == 'yes','is_fraud'] = '1'
merged["is_fraud"] = pd.to_numeric(merged["is_fraud"], errors="coerce").astype("Int64")

merged = add_missing_flags(merged, ["is_fraud"])

#merged.to_csv("merged.csv", index=False)

# 先刪除不需要的DataFrame以節省記憶體
del transactions_df, users_df, cards_df, train_fraud_labels_df, cols_to_encode, dummies_cards, dummies_use, dummies_gender

/var/folders/yb/xnfk9z6x34527z3924bcjqcr0000gn/T/ipykernel_27178/933642251.py:16: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  merged["is_fraud"] = merged["is_fraud"].astype(str)
/var/folders/yb/xnfk9z6x34527z3924bcjqcr0000gn/T/ipykernel_27

In [30]:
#backup_merged = merged.copy()

# 04_RFM + DK 重點微調

In [ ]:
#merged = backup_merged.copy() ##有出事再趕快回復原狀

## 04_RFM features engineering model

### 04-1_RFM features 新增

In [31]:
import numpy as np
import pandas as pd

# 確保日期是 datetime 並排序
merged['date'] = pd.to_datetime(merged['date'])
merged = merged.sort_values(by=['client_id_x', 'date']).reset_index(drop=True)

# --- RecencyInterval ---
merged['RecencyInterval'] = merged.groupby('client_id_x')['date'].diff().dt.total_seconds().fillna(0)/60

# --- TxnFrequency for multiple windows (向量化滑動窗口) ---
window_days = [7, 30, 60, 90]
for w in window_days:
    merged[f'TxnFrequency_{w}d'] = 0

def compute_freq_vectorized(dates, windows):
    """向量化計算每筆交易在每個 window 內的交易數"""
    n = len(dates)
    dates_int = dates.values.astype('datetime64[D]').astype(int)
    res = {w: np.zeros(n, dtype=int) for w in windows}
    for w in windows:
        left = 0
        counts = np.zeros(n, dtype=int)
        for right in range(n):
            while dates_int[right] - dates_int[left] > w:
                left += 1
            counts[right] = right - left + 1
        res[w] = counts
    return res

# 分組計算
for cid, g in merged.groupby('client_id_x', sort=False):
    freq_dict = compute_freq_vectorized(g['date'], window_days)
    for w in window_days:
        merged.loc[g.index, f'TxnFrequency_{w}d'] = freq_dict[w]

# --- AmtDelta ---
merged['prev_amount'] = merged.groupby('client_id_x')['amount'].shift(1)
merged['AmtDelta'] = merged['amount'] - merged['prev_amount']
merged['AmtDelta'] = merged['AmtDelta'].fillna(0)
merged.drop(columns='prev_amount', inplace=True)

/var/folders/yb/xnfk9z6x34527z3924bcjqcr0000gn/T/ipykernel_27178/781920426.py:5: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  merged['date'] = pd.to_datetime(merged['date'])
/var/folders/yb/xnfk9z6x34527z3924bcjqcr0000gn/T/ipykernel_27178/7

### 04-3_DK features 新增

In [32]:
# US region mapping
us_region_map = {
    'Northeast': ['NY','NJ','PA','MA','CT','RI','NH','VT','ME'],
    'Midwest': ['IL','OH','MI','IN','WI','MN','IA','MO','ND','SD','NE','KS'],
    'South': ['FL','GA','SC','NC','AL','MS','LA','TX','OK','TN','KY','VA','WV','AR','MD','DE','DC'],
    'West': ['CA','WA','OR','NV','AZ','NM','CO','UT','ID','MT','WY','AK','HI'],
}
continent_map = {
    'Europe': [ ... ],  # 原本 continent_map['Europe'] 可直接使用
    'Online': ['online','AA']
}

us_region_lookup = {state: region for region, states in us_region_map.items() for state in states}

# --- 向量化 location 特徵 ---
merged['merchant_online'] = merged['merchant_state'].eq('online').astype('uint8')
merged['merchant_us'] = merged['merchant_state'].isin(us_region_lookup.keys()).astype('uint8')
merged['merchant_eu'] = merged['merchant_state'].isin(continent_map['Europe']).astype('uint8')
merged['merchant_others'] = (~merged[['merchant_online','merchant_us','merchant_eu']].any(axis=1)).astype('uint8')

# --- 首次交易標記 ---
merged['FirstTxnInRegion'] = (~merged.duplicated(subset=['client_id_x', 'merchant_state'])).astype('uint8')

# DifferentState
merged['prev_state']=(merged
                     .groupby('client_id_x')['merchant_state']
                     .shift(1))

merged['DifferentState'] = (
    (merged['merchant_state'] != merged['prev_state'])
    & merged['prev_state'].notna()
).astype(int)

# create txn_to_limit_ratio
# set 0 if credit_limit is 0
merged['TxnToLimitRatio'] = merged['amount'] / merged['credit_limit']
merged['TxnToLimitRatio'] = merged['TxnToLimitRatio'].replace([np.inf, -np.inf], 0).fillna(0)

merged = merged.drop(columns=['prev_state'])


/var/folders/yb/xnfk9z6x34527z3924bcjqcr0000gn/T/ipykernel_27178/1203057931.py:16: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  merged['merchant_online'] = merged['merchant_state'].eq('online').astype('uint8')
/var/folders/yb/xnfk9z6x34527z

### 04-3_資料進行變數轉換以求模型配飾更佳表現(不一樣)

In [33]:
#merged[["card_id","card_number"]]
import numpy as np
from scipy import stats 

# === (1) log轉換 ===
merged['amount'] = np.where(merged['amount'] < 0, 0, merged['amount'])  # 負數變 0
merged['amount'] = np.log(merged['amount'] + 1)  

# === (3) 平方根轉換 ===
merged['credit_limit']=np.sqrt(merged['credit_limit'])
merged['total_debt']=np.sqrt(merged['total_debt'])

# === (3) 立方根轉換 ===
merged['yearly_income']=np.cbrt(merged['yearly_income'])
merged['per_capita_income']=np.cbrt(merged['per_capita_income'])

## Box-Cox Transformation
###merged['yearly_income'], fitted_lambda = stats.boxcox(merged['yearly_income'])

# === (5) Yeo–Johnson 轉換（可處理負值） ===
###merged['per_capita_income'], lambdaValue =stats.yeojohnson(merged['per_capita_income'])

/var/folders/yb/xnfk9z6x34527z3924bcjqcr0000gn/T/ipykernel_27178/2574060425.py:6: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  merged['amount'] = np.where(merged['amount'] < 0, 0, merged['amount'])  # 負數變 0
/var/folders/yb/xnfk9z6x34527z392

### 04-4_分割訓練集及測試集、不同block 並從這裡才計算 HighRiskMCC 避免資料洩漏

In [34]:
import pandas as pd
import numpy as np

# ==========================================================
# 0. merged 已存在
#    必須至少包含：
#    - date
#    - is_fraud
#    - mcc_code
# ==========================================================

# 例如：
# merged = pd.read_csv("your_data.csv", parse_dates=["date"])


# ==========================================================
# 1. 前處理
# ==========================================================
# --- 選取數值型變數 ---
num_cols = merged.select_dtypes(
    include=['int64', 'float64', 'uint8', 'datetime64[ns]']
).columns

df2 = merged[num_cols].copy()

# --- dropna ---
df_cleaned = df2.dropna().copy()
del df2

# --- 避免共線性 ---
drop_cols = [
    "is_fraud_missing_flag",
    "card_type_Debit (Prepaid)",
    "card_brand_Discover",
    "use_chip_Online Transaction"
]
existing_drop_cols = [c for c in drop_cols if c in df_cleaned.columns]
df_cleaned.drop(columns=existing_drop_cols, inplace=True)

# --- 確保 date 欄位在 df_cleaned 中 ---
if 'date' not in df_cleaned.columns:
    df_cleaned['date'] = merged.loc[df_cleaned.index, 'date']

# --- 確保 date 為 datetime ---
df_cleaned['date'] = pd.to_datetime(df_cleaned['date'], errors='coerce')

# date 若轉換失敗，再清一次
df_cleaned = df_cleaned.dropna(subset=['date']).copy()

# --- 依時間排序 ---
df_sorted = df_cleaned.sort_values('date').copy()
df_sorted['year'] = df_sorted['date'].dt.year

# ==========================================================
# 2. 真正的「2年一組」time_block
#    2010-2011
#    2012-2013
#    2014-2015 ...
# ==========================================================
start_year = df_sorted['year'].min()

# 先算每列屬於第幾個2年區塊
block_num = ((df_sorted['year'] - start_year) // 2).astype(int)

block_start = start_year + block_num * 2
block_end = block_start + 1

df_sorted['time_block'] = block_start.astype(str) + '-' + block_end.astype(str)

print("=== Time blocks ===")
print(df_sorted['time_block'].value_counts().sort_index())


# ==========================================================
# 3. 定義 function：只用 training data 算 HighRiskMCC
# ==========================================================
def add_high_risk_mcc(train_df, apply_dfs, threshold=0.02):
    """
    train_df: 用來計算 fraud rate 的 training dataframe
    apply_dfs: list of dataframes，要套用 HighRiskMCC 的資料
    threshold: fraud rate 門檻
    """
    if 'mcc_code' not in train_df.columns:
        raise ValueError("缺少 'mcc_code' 欄位")
    if 'is_fraud' not in train_df.columns:
        raise ValueError("缺少 'is_fraud' 欄位")

    fraud_rate = train_df.groupby('mcc_code')['is_fraud'].mean()
    high_risk_mcc = fraud_rate[fraud_rate > threshold].index

    output_dfs = []
    for df in apply_dfs:
        df_new = df.copy()
        df_new['HighRiskMCC'] = df_new['mcc_code'].isin(high_risk_mcc).astype('uint8')
        output_dfs.append(df_new)

    return output_dfs, high_risk_mcc


# ==========================================================
# 4. 建立 outer / inner Nested CV splits
# ==========================================================
all_blocks = sorted(df_sorted['time_block'].unique())
print("\nAll ordered blocks:", all_blocks)

nested_cv_data = []

# outer 至少要留：
# - 前面至少 2 個 block 給 train_outer
# - 1 個 block 當 test_outer
for outer_idx in range(2, len(all_blocks)):
    train_outer_blocks = all_blocks[:outer_idx]
    test_outer_block = all_blocks[outer_idx]

    train_outer_raw = df_sorted[df_sorted['time_block'].isin(train_outer_blocks)].copy()
    test_outer_raw = df_sorted[df_sorted['time_block'] == test_outer_block].copy()

    print("\n" + "=" * 70)
    print(f"OUTER FOLD {outer_idx - 1}")
    print(f"Train outer blocks: {train_outer_blocks}")
    print(f"Test outer block : {test_outer_block}")
    print(f"train_outer_raw shape = {train_outer_raw.shape}")
    print(f"test_outer_raw  shape = {test_outer_raw.shape}")

    # -----------------------------
    # outer feature engineering
    # 只用 train_outer 算 HighRiskMCC，再套到 train_outer / test_outer
    # -----------------------------
    [train_outer_fe, test_outer_fe], outer_high_risk_mcc = add_high_risk_mcc(
        train_df=train_outer_raw,
        apply_dfs=[train_outer_raw, test_outer_raw],
        threshold=0.02
    )

    # -----------------------------
    # inner loop
    # 在 train_outer_blocks 裡做 expanding validation
    # 例如：
    # blocks = [2010-2011, 2012-2013, 2014-2015]
    # inner:
    #   train=[2010-2011], val=[2012-2013]
    #   train=[2010-2011,2012-2013], val=[2014-2015]
    # -----------------------------
    inner_results = []

    for inner_idx in range(1, len(train_outer_blocks)):
        train_inner_blocks = train_outer_blocks[:inner_idx]
        val_inner_block = train_outer_blocks[inner_idx]

        train_inner_raw = train_outer_raw[
            train_outer_raw['time_block'].isin(train_inner_blocks)
        ].copy()

        val_inner_raw = train_outer_raw[
            train_outer_raw['time_block'] == val_inner_block
        ].copy()

        # inner feature engineering
        [train_inner_fe, val_inner_fe], inner_high_risk_mcc = add_high_risk_mcc(
            train_df=train_inner_raw,
            apply_dfs=[train_inner_raw, val_inner_raw],
            threshold=0.02
        )

        # 最後把不能進模型的欄位先去掉
        drop_final_cols = ['date', 'year', 'time_block']

        train_inner_final = train_inner_fe.drop(
            columns=[c for c in drop_final_cols if c in train_inner_fe.columns]
        ).copy()

        val_inner_final = val_inner_fe.drop(
            columns=[c for c in drop_final_cols if c in val_inner_fe.columns]
        ).copy()

        inner_results.append({
            'inner_fold': inner_idx,
            'train_blocks': train_inner_blocks,
            'val_block': val_inner_block,
            'train_inner': train_inner_final,
            'val_inner': val_inner_final,
            'high_risk_mcc_list': list(inner_high_risk_mcc)
        })

        print("\n  --- INNER FOLD", inner_idx, "---")
        print("  train_inner_blocks:", train_inner_blocks)
        print("  val_inner_block   :", val_inner_block)
        print("  train_inner shape :", train_inner_final.shape)
        print("  val_inner shape   :", val_inner_final.shape)
        print("  train fraud count:")
        print(train_inner_final['is_fraud'].value_counts(dropna=False))
        print("  val fraud count:")
        print(val_inner_final['is_fraud'].value_counts(dropna=False))

    # outer final datasets
    drop_final_cols = ['date', 'year', 'time_block']

    train_outer_final = train_outer_fe.drop(
        columns=[c for c in drop_final_cols if c in train_outer_fe.columns]
    ).copy()

    test_outer_final = test_outer_fe.drop(
        columns=[c for c in drop_final_cols if c in test_outer_fe.columns]
    ).copy()

    print("\n  >>> OUTER FINAL DATA <<<")
    print("  train_outer shape:", train_outer_final.shape)
    print("  test_outer shape :", test_outer_final.shape)
    print("  train_outer fraud count:")
    print(train_outer_final['is_fraud'].value_counts(dropna=False))
    print("  test_outer fraud count:")
    print(test_outer_final['is_fraud'].value_counts(dropna=False))

    nested_cv_data.append({
        'outer_fold': outer_idx - 1,
        'train_outer_blocks': train_outer_blocks,
        'test_outer_block': test_outer_block,
        'train_outer': train_outer_final,
        'test_outer': test_outer_final,
        'inner_results': inner_results,
        'high_risk_mcc_list_outer': list(outer_high_risk_mcc)
    })


# ==========================================================
# 5. 範例：怎麼取某一 fold 的資料
# ==========================================================
print("\n" + "=" * 70)
print("Nested CV building finished.")
print(f"Total outer folds: {len(nested_cv_data)}")

if len(nested_cv_data) > 0:
    fold0 = nested_cv_data[0]

    print("\nExample access:")
    print("fold0 keys:", fold0.keys())
    print("fold0 outer_fold:", fold0['outer_fold'])
    print("fold0 train_outer_blocks:", fold0['train_outer_blocks'])
    print("fold0 test_outer_block:", fold0['test_outer_block'])
    print("fold0 train_outer shape:", fold0['train_outer'].shape)
    print("fold0 test_outer shape:", fold0['test_outer'].shape)

    if len(fold0['inner_results']) > 0:
        inner0 = fold0['inner_results'][0]
        print("\nFirst inner fold example:")
        print("inner0 train_blocks:", inner0['train_blocks'])
        print("inner0 val_block:", inner0['val_block'])
        print("inner0 train_inner shape:", inner0['train_inner'].shape)
        print("inner0 val_inner shape:", inner0['val_inner'].shape)

/var/folders/yb/xnfk9z6x34527z3924bcjqcr0000gn/T/ipykernel_27178/3905514107.py:45: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  df_cleaned['date'] = pd.to_datetime(df_cleaned['date'], errors='coerce')
/var/folders/yb/xnfk9z6x34527z3924bcjqc

=== Time blocks ===
time_block
2010-2011    1694957
2012-2013    1792725
2014-2015    1845297
2016-2017    1870046
2018-2019    1711938
Name: count, dtype: int64

All ordered blocks: ['2010-2011', '2012-2013', '2014-2015', '2016-2017', '2018-2019']

OUTER FOLD 1
Train outer blocks: ['2010-2011', '2012-2013']
Test outer block : 2014-2015
train_outer_raw shape = (3487682, 49)
test_outer_raw  shape = (1845297, 49)


/var/folders/yb/xnfk9z6x34527z3924bcjqcr0000gn/T/ipykernel_27178/3905514107.py:94: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  df_new['HighRiskMCC'] = df_new['mcc_code'].isin(high_risk_mcc).astype('uint8')
/var/folders/yb/xnfk9z6x34527z392


  --- INNER FOLD 1 ---
  train_inner_blocks: ['2010-2011']
  val_inner_block   : 2012-2013
  train_inner shape : (1694957, 47)
  val_inner shape   : (1792725, 47)
  train fraud count:
is_fraud
0    1692347
1       2610
Name: count, dtype: Int64
  val fraud count:
is_fraud
0    1790465
1       2260
Name: count, dtype: Int64

  >>> OUTER FINAL DATA <<<
  train_outer shape: (3487682, 47)
  test_outer shape : (1845297, 47)
  train_outer fraud count:
is_fraud
0    3482812
1       4870
Name: count, dtype: Int64
  test_outer fraud count:
is_fraud
0    1842444
1       2853
Name: count, dtype: Int64

OUTER FOLD 2
Train outer blocks: ['2010-2011', '2012-2013', '2014-2015']
Test outer block : 2016-2017
train_outer_raw shape = (5332979, 49)
test_outer_raw  shape = (1870046, 49)


/var/folders/yb/xnfk9z6x34527z3924bcjqcr0000gn/T/ipykernel_27178/3905514107.py:94: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  df_new['HighRiskMCC'] = df_new['mcc_code'].isin(high_risk_mcc).astype('uint8')
/var/folders/yb/xnfk9z6x34527z392

...


  --- INNER FOLD 1 ---
  train_inner_blocks: ['2010-2011']
  val_inner_block   : 2012-2013
  train_inner shape : (1694957, 47)
  val_inner shape   : (1792725, 47)
  train fraud count:
is_fraud
0    1692347
1       2610
Name: count, dtype: Int64
  val fraud count:
is_fraud
0    1790465
1       2260
Name: count, dtype: Int64


/var/folders/yb/xnfk9z6x34527z3924bcjqcr0000gn/T/ipykernel_27178/3905514107.py:94: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  df_new['HighRiskMCC'] = df_new['mcc_code'].isin(high_risk_mcc).astype('uint8')
/var/folders/yb/xnfk9z6x34527z392


  --- INNER FOLD 2 ---
  train_inner_blocks: ['2010-2011', '2012-2013']
  val_inner_block   : 2014-2015
  train_inner shape : (3487682, 47)
  val_inner shape   : (1845297, 47)
  train fraud count:
is_fraud
0    3482812
1       4870
Name: count, dtype: Int64
  val fraud count:
is_fraud
0    1842444
1       2853
Name: count, dtype: Int64


/var/folders/yb/xnfk9z6x34527z3924bcjqcr0000gn/T/ipykernel_27178/3905514107.py:94: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  df_new['HighRiskMCC'] = df_new['mcc_code'].isin(high_risk_mcc).astype('uint8')
/var/folders/yb/xnfk9z6x34527z392


  --- INNER FOLD 3 ---
  train_inner_blocks: ['2010-2011', '2012-2013', '2014-2015']
  val_inner_block   : 2016-2017
  train_inner shape : (5332979, 47)
  val_inner shape   : (1870046, 47)
  train fraud count:
is_fraud
0    5325256
1       7723
Name: count, dtype: Int64
  val fraud count:
is_fraud
0    1867426
1       2620
Name: count, dtype: Int64

  >>> OUTER FINAL DATA <<<
  train_outer shape: (7203025, 47)
  test_outer shape : (1711938, 47)
  train_outer fraud count:
is_fraud
0    7192682
1      10343
Name: count, dtype: Int64
  test_outer fraud count:
is_fraud
0    1708949
1       2989
Name: count, dtype: Int64

Nested CV building finished.
Total outer folds: 3

Example access:
fold0 keys: dict_keys(['outer_fold', 'train_outer_blocks', 'test_outer_block', 'train_outer', 'test_outer', 'inner_results', 'high_risk_mcc_list_outer'])
fold0 outer_fold: 1
fold0 train_outer_blocks: ['2010-2011', '2012-2013']
fold0 test_outer_block: 2014-2015
fold0 train_outer shape: (3487682, 47)
fold0 t

### 04-5_定義不同的features group 方便模型運算

In [35]:
'''
result = pd.DataFrame(columns=[
    "Model", "Features", 
    "Train AUC", "Test AUC", 
    "Train PR AUC", "Test PR AUC"
])

'''

# ALL features

all_cols = ['transaction_id', 'date', 'client_id_x', 'card_id', 'amount',
       'merchant_id', 'merchant_city', 'merchant_state', 'zip', 'mcc_code',
       'errors', 'merchant_state_missing_flag', 'zip_missing_flag',
       'errors_missing_flag', 'use_chip_Chip Transaction',
       'use_chip_Online Transaction', 'use_chip_Swipe Transaction',
       'current_age', 'retirement_age', 'birth_year', 'birth_month', 'gender',
       'address', 'latitude', 'longitude', 'per_capita_income',
       'yearly_income', 'total_debt', 'credit_score', 'num_credit_cards',
       'gender_Male', 'card_brand', 'card_type', 'card_number', 'expires',
       'cvv', 'num_cards_issued', 'credit_limit', 'acct_open_date',
       'year_pin_last_changed', 'card_on_dark_web', 'card_type_Credit',
       'card_type_Debit', 'card_type_Debit (Prepaid)', 'card_brand_Amex',
       'card_brand_Discover', 'card_brand_Mastercard', 'card_brand_Visa',
       'has_chip_YES', 'is_fraud_missing_flag']

VIF_col = ["is_fraud_missing_flag","card_type_Debit (Prepaid)", 
                         "card_brand_Discover", "use_chip_Online Transaction",'is_fraud_missing_flag','merchant_state_missing_flag', 'zip_missing_flag','card_on_dark_web','errors_missing_flag']

identifier = [
    'transaction_id',
    'client_id_x',
    'card_id',
    'card_number',
    'cvv'
]

leakage_cols = ['latitude', 'longitude', 'zip','merchant_id','mcc_code'] #gemini建議


set_VIF = set(VIF_col)
set_identifier =set(identifier)
exclude_cols = set(VIF_col) | set(identifier)| set(leakage_cols)
all_cols = [x for x in all_cols if x not in exclude_cols]


# RFM features
rfm_cols = [
    'RecencyInterval', 'TxnFrequency_7d','TxnFrequency_30d',
    'TxnFrequency_60d', 'TxnFrequency_90d', 'AmtDelta','TxnToLimitRatio'
]

# DK features
dk_cols = [
    'merchant_online', 'merchant_us', 'merchant_eu', 'merchant_others',
    'FirstTxnInRegion','HighRiskMCC','DifferentState'
]

# Grouping
feature_groups = {
    "X_all": all_cols,
    "X_rfm": rfm_cols,
    "X_dk": dk_cols,
    "X_all + X_rfm": all_cols + rfm_cols,
    "X_all + X_dk": all_cols + dk_cols,
    "X_rfm + X_dk": rfm_cols + dk_cols,
    "X_all + X_rfm + X_dk": all_cols + rfm_cols + dk_cols
}

### 04-6_Assumption: Avoid Multicollinearity（老師建議我們先省略）

In [ ]:
'''
##處理高度共線性變數
train_df.drop(columns=["per_capita_income"], inplace=True)
train_df.drop(columns=["use_chip_Chip Transaction","merchant_state_missing_flag","zip_missing_flag"], inplace=True)           
train_df.drop(columns=["card_brand_Visa" ,"card_brand_Amex","card_type_Credit"], inplace=True)

test_df.drop(columns=["per_capita_income"], inplace=True)
test_df.drop(columns=["use_chip_Chip Transaction","merchant_state_missing_flag","zip_missing_flag"], inplace=True)           
test_df.drop(columns=["card_brand_Visa" ,"card_brand_Amex","card_type_Credit"], inplace=True)
'''

# 05_Features Selection 部分

## 05-1_Stepwise selection

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, average_precision_score
import warnings

# 忽略 statsmodels 可能產生的迭代警告，保持輸出乾淨
warnings.filterwarnings("ignore")

# ===========================================================
# 1. 核心函數：嚴謹版 Stepwise (Forward + Backward)
# ===========================================================

def check_data_validity(y_data, stage="Train"):
    """檢查資料是否足夠進行 Logistic Regression"""
    if len(y_data) == 0:
        return False, f"{stage} set is empty"
    if y_data.nunique() < 2:
        val = y_data.iloc[0] if len(y_data) > 0 else "None"
        return False, f"{stage} set has only 1 class (value: {val})"
    return True, ""

def run_stepwise_logit(train_df, test_df, feature_cols, dep_var="is_fraud", 
                       threshold_in=0.01, threshold_out=0.05, max_iter=50):
    """
    Stepwise Selection based on P-values (Forward Selection + Backward Elimination)
    """
    # --- 1. 準備數據 ---
    X_train = train_df[feature_cols].fillna(0)
    y_train = train_df[dep_var]
    X_test  = test_df[feature_cols].fillna(0)
    y_test  = test_df[dep_var]

    # --- 2. 防崩潰檢查 (Edge Case Handling) ---
    is_valid_train, msg_train = check_data_validity(y_train, "Train")
    if not is_valid_train:
        return {"status": "skip", "message": msg_train}
    
    # 測試集若無效，模型仍可訓練，只是無法算 Test AUC
    is_valid_test, _ = check_data_validity(y_test, "Test")

    # --- 3. 標準化 (Standardization) ---
    # Logistic 對變數尺度極其敏感，Stepwise 前務必標準化
    try:
        scaler = StandardScaler()
        X_train_std = pd.DataFrame(scaler.fit_transform(X_train), columns=feature_cols, index=train_df.index)
        X_test_std  = pd.DataFrame(scaler.transform(X_test), columns=feature_cols, index=test_df.index)
    except Exception as e:
        return {"status": "error", "message": f"Scaling failed: {str(e)}"}

    # --- 4. 迭代篩選 ---
    included = []
    candidates = list(feature_cols)

    for i in range(max_iter):
        changed = False
        
        # ====== Forward Step: 嘗試加入一個最好的變數 ======
        best_pval = 1.0
        best_feature = None
        
        for candidate in candidates:
            try:
                # 暫時加入候選變數
                current_vars = included + [candidate]
                X_const = sm.add_constant(X_train_std[current_vars])
                
                # 訓練 (disp=0 不印出 log)
                model = sm.Logit(y_train, X_const).fit(disp=0, method='newton')
                
                # 取得該變數的 p-value
                pval = model.pvalues[candidate]
                
                if pval < best_pval:
                    best_pval = pval
                    best_feature = candidate
            except:
                # 若發生 Singular Matrix (矩陣奇異) 或收斂失敗，跳過該變數
                continue
        
        # 若 P-value 小於門檻則加入
        if best_feature is not None and best_pval < threshold_in:
            included.append(best_feature)
            candidates.remove(best_feature)
            changed = True
            # print(f"    + Add: {best_feature} (p={best_pval:.4f})")

        # ====== Backward Step: 嘗試移除一個最差的變數 ======
        if included:
            try:
                X_const = sm.add_constant(X_train_std[included])
                model = sm.Logit(y_train, X_const).fit(disp=0)
                
                # 找出 p-value 最大的變數 (排除 const)
                pvalues = model.pvalues.drop('const', errors='ignore')
                if not pvalues.empty:
                    worst_pval = pvalues.max()
                    worst_feature = pvalues.idxmax()
                    
                    # 若 P-value 大於門檻則移除
                    if worst_pval > threshold_out:
                        included.remove(worst_feature)
                        candidates.append(worst_feature)
                        changed = True
                        # print(f"    - Drop: {worst_feature} (p={worst_pval:.4f})")
            except:
                pass

        if not changed:
            break

    # --- 5. 最終結果回傳 ---
    if not included:
        return {"status": "skip", "message": "No variables selected (all insignificant)"}

    # 計算當下 Stepwise 模型的 Metrics (供參考)
    try:
        X_train_final = sm.add_constant(X_train_std[included])
        final_model = sm.Logit(y_train, X_train_final).fit(disp=0)
        
        train_prob = final_model.predict(X_train_final)
        train_auc = roc_auc_score(y_train, train_prob)
        train_pr  = average_precision_score(y_train, train_prob)
        
        test_auc = np.nan
        test_pr = np.nan
        
        if is_valid_test:
            try:
                X_test_final = sm.add_constant(X_test_std[included])
                test_prob = final_model.predict(X_test_final)
                test_auc = roc_auc_score(y_test, test_prob)
                test_pr  = average_precision_score(y_test, test_prob)
            except:
                pass

        return {
            "status": "success",
            "selected_features": included,  # <--- 重點：選出的變數列表
            "metrics": {
                "Train AUC": round(train_auc, 4),
                "Test AUC": round(test_auc, 4) if not np.isnan(test_auc) else np.nan,
                "Train PR-AUC": round(train_pr, 4),
                "Test PR-AUC": round(test_pr, 4) if not np.isnan(test_pr) else np.nan,
                "Num Features": len(included)
            }
        }
    except Exception as e:
        return {"status": "error", "message": str(e)}


# ===========================================================
# 2. 設定與特徵群組
# ===========================================================

# 確保 df_sorted 存在
if 'df_sorted' not in locals():
    raise ValueError("⚠️ Error: 'df_sorted' 未定義，請先載入您的資料。")

# 定義要排除的欄位
identifier = ['transaction_id', 'client_id_x', 'card_id', 'card_number', 'cvv']
exclude_cols = set(identifier) | {'is_fraud', 'date', 'year', 'time_block', 'HighRiskMCC'}

def clean_cols(cols):
    """過濾掉 ID 與 Target 等不應放入模型的欄位"""
    return [c for c in cols if c not in exclude_cols]

# 定義您的 Feature Groups
# (假設 all_cols, rfm_cols, dk_cols 變數已存在於您的環境中)
feature_groups = {
    "X_all": clean_cols(all_cols),
    "X_rfm": clean_cols(rfm_cols),
    "X_dk": clean_cols(dk_cols),
    "X_all + X_rfm": clean_cols(all_cols + rfm_cols),
    "X_all + X_dk": clean_cols(all_cols + dk_cols),
    "X_rfm + X_dk": clean_cols(rfm_cols + dk_cols),
    "X_all + X_rfm + X_dk": clean_cols(all_cols + rfm_cols + dk_cols)
}


# ===========================================================
# 3. 主執行迴圈 (儲存變數 + 產出初步報表)
# ===========================================================

# ★ 這是最重要的儲存容器：[Block ID][Group Name] -> [變數列表]
stepwise_feature_storage = {} 
stepwise_report_list = []

for block_id, block_df in df_sorted.groupby('time_block'):
    
    current_year = 2010 + block_id
    print(f"\n{'='*60}")
    print(f"🧩 Stepwise Selection for Block {block_id} (Year: {current_year})")
    print(f"{'='*60}")

    # --- 1. 時間序列切分 (Temporal Split) ---
    block_df = block_df.sort_values('date')
    split_index = int(len(block_df) * 0.8)
    
    train_raw = block_df.iloc[:split_index].copy()
    test_raw  = block_df.iloc[split_index:].copy()

    # --- 2. 特徵工程 (Anti-Leakage) ---
    # 必須在切分後才做 Risk 計算
    try:
        fraud_rate = train_raw.groupby('mcc_code')['is_fraud'].mean()
        high_risk_mcc = fraud_rate[fraud_rate > 0.02].index
        train_raw['HighRiskMCC'] = train_raw['mcc_code'].isin(high_risk_mcc).astype('uint8')
        test_raw['HighRiskMCC']  = test_raw['mcc_code'].isin(high_risk_mcc).astype('uint8')
    except:
        pass # 若 train 空了會被後面的 check 擋下，這裡忽略

    # 初始化該年份的儲存空間
    stepwise_feature_storage[block_id] = {}

    # --- 3. 針對每個 Group 跑 Stepwise ---
    for group_name, feature_list in feature_groups.items():
        
        # 確保特徵存在
        valid_features = [f for f in feature_list if f in train_raw.columns]
        if not valid_features:
            continue
            
        print(f"   🔸 Group: {group_name} (Pool: {len(valid_features)})...", end=" ")

        # 執行 Stepwise
        res = run_stepwise_logit(
            train_raw, 
            test_raw, 
            feature_cols=valid_features,
            threshold_in=0.01,   # 顯著才入選
            threshold_out=0.05   # 不顯著就踢除
        )

        # 處理結果
        if res["status"] == "success":
            m = res["metrics"]
            selected_feats = res["selected_features"]
            
            print(f"✅ Selected {len(selected_feats)} feats | Test AUC: {m['Test AUC']}")
            
            # (A) 存變數 (給後續模型用)
            stepwise_feature_storage[block_id][group_name] = selected_feats
            
            # (B) 存報表數據
            row = {
                "Year": current_year,
                "Feature Group": group_name,
                "Train AUC": m["Train AUC"], "Test AUC": m["Test AUC"],
                "Train PR-AUC": m["Train PR-AUC"], "Test PR-AUC": m["Test PR-AUC"],
                "Num Features": m["Num Features"]
            }
            stepwise_report_list.append(row)
            
        elif res["status"] == "skip":
            print(f"⚠️ Skipped: {res['message']}")
            stepwise_feature_storage[block_id][group_name] = [] # 存空 list
            # 存 NaN 報表
            row = {
                "Year": current_year, "Feature Group": group_name,
                "Train AUC": np.nan, "Test AUC": np.nan,
                "Train PR-AUC": np.nan, "Test PR-AUC": np.nan,
                "Num Features": 0
            }
            stepwise_report_list.append(row)
            
        else:
            print(f"❌ Error: {res['message']}")
            stepwise_feature_storage[block_id][group_name] = []

print("\n🎉 Stepwise Selection Completed!")
print(f"變數結果已儲存於 'stepwise_feature_storage' 字典中。")


# ===========================================================
# 4. 顯示 Stepwise 初步結果報表
# ===========================================================
if stepwise_report_list:
    df_sw_raw = pd.DataFrame(stepwise_report_list)
    
    # 轉置表格 (Pivot) 以符合您習慣的格式
    target_metrics = ["Train AUC", "Test AUC", "Train PR-AUC", "Test PR-AUC"]
    df_sw_pivot = df_sw_raw.pivot(index="Feature Group", columns="Year", values=target_metrics)
    
    # 調整欄位順序
    df_sw_pivot.columns = df_sw_pivot.columns.swaplevel(0, 1)
    unique_years = sorted(df_sw_raw["Year"].unique())
    ordered_columns = [(y, m) for y in unique_years for m in target_metrics]
    
    df_sw_final = df_sw_pivot.reindex(columns=ordered_columns)
    
    print("\n=== Stepwise Logistic Regression 變數篩選成效表 ===")
    print(df_sw_final.to_string())
else:
    print("No data to report.")

In [ ]:
## stepwise變數選取 結果

import pickle

# 1. 儲存 (Save)
with open('stepwise_feature_storage.pkl', 'wb') as f:
    pickle.dump(stepwise_feature_storage, f)

print("✅ 已成功儲存為 stepwise_feature_storage.pkl")

# ==========================================
# 下次要用時，執行這段讀取 (Load)
# ==========================================
# with open('stepwise_feature_storage.pkl', 'rb') as f:
#     stepwise_feature_storage = pickle.load(f)
# print("✅ 已讀取變數設定")

## 05-2_Elast Net

## 05-3_SHAP

# 06_Modeling (LR, XGB and LGBM)

## 06-1 Logistic Regression

#### (6-1 a) Logistic function定義

In [36]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score

# ===========================================================
# 1. Full Logistic Regression 訓練函數
# ===========================================================
def fit_full_logit(train_df, test_df, feature_cols, dep_var="is_fraud"):
    """
    針對給定的特徵列表進行全變數 Logistic Regression (L2 Penalty)
    """
    # 準備 X 和 y
    X_train = train_df[feature_cols].copy()
    y_train = train_df[dep_var].copy()

    X_test = test_df[feature_cols].copy()
    y_test = test_df[dep_var].copy()

    # 處理缺失值
    X_train = X_train.fillna(0)
    X_test = X_test.fillna(0)

    # 標準化
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    # 訓練模型
    model = LogisticRegression(
        penalty="l2",
        solver="lbfgs",
        max_iter=2000,
        n_jobs=-1,
        random_state=42
    )

    try:
        model.fit(X_train_scaled, y_train)

        # 預測機率
        train_pred_prob = model.predict_proba(X_train_scaled)[:, 1]
        test_pred_prob = model.predict_proba(X_test_scaled)[:, 1]

        # 計算指標
        train_auc = roc_auc_score(y_train, train_pred_prob)
        test_auc = roc_auc_score(y_test, test_pred_prob)

        train_prauc = average_precision_score(y_train, train_pred_prob)
        test_prauc = average_precision_score(y_test, test_pred_prob)

        return {
            "status": "success",
            "model": model,
            "scaler": scaler,
            "metrics": {
                "Train AUC": round(train_auc, 4),
                "Test AUC": round(test_auc, 4),
                "Train PR-AUC": round(train_prauc, 4),
                "Test PR-AUC": round(test_prauc, 4)
            }
        }

    except Exception as e:
        return {"status": "error", "message": str(e)}


# ===========================================================
# 2. 準備 Feature Groups
# ===========================================================
identifier = ['transaction_id', 'client_id_x', 'card_id', 'card_number', 'cvv']
exclude_cols = set(identifier) | {'is_fraud', 'date', 'year', 'time_block'}

def clean_cols(cols):
    # 去掉 exclude_cols，並保持原順序去重複
    seen = set()
    cleaned = []
    for c in cols:
        if c not in exclude_cols and c not in seen:
            cleaned.append(c)
            seen.add(c)
    return cleaned

feature_groups = {
    "X_all": clean_cols(all_cols),
    "X_rfm": clean_cols(rfm_cols),
    "X_dk": clean_cols(dk_cols),
    "X_all + X_rfm": clean_cols(all_cols + rfm_cols),
    "X_all + X_dk": clean_cols(all_cols + dk_cols),
    "X_rfm + X_dk": clean_cols(rfm_cols + dk_cols),
    "X_all + X_rfm + X_dk": clean_cols(all_cols + rfm_cols + dk_cols)
}


# ===========================================================
# 3. 建立 outer / inner Nested CV splits
# ===========================================================
def add_high_risk_mcc(train_df, apply_dfs, threshold=0.02):
    """
    train_df: 用來計算 fraud rate 的 training dataframe
    apply_dfs: list of dataframes，要套用 HighRiskMCC 的資料
    threshold: fraud rate 門檻
    """
    if 'mcc_code' not in train_df.columns:
        raise ValueError("缺少 'mcc_code' 欄位")
    if 'is_fraud' not in train_df.columns:
        raise ValueError("缺少 'is_fraud' 欄位")

    fraud_rate = train_df.groupby('mcc_code')['is_fraud'].mean()
    high_risk_mcc = fraud_rate[fraud_rate > threshold].index

    output_dfs = []
    for df in apply_dfs:
        df_new = df.copy()
        df_new['HighRiskMCC'] = df_new['mcc_code'].isin(high_risk_mcc).astype('uint8')
        output_dfs.append(df_new)

    return output_dfs, high_risk_mcc


# df_sorted 需已先存在
all_blocks = sorted(df_sorted['time_block'].unique())
nested_cv_data = []

for outer_idx in range(2, len(all_blocks)):
    train_outer_blocks = all_blocks[:outer_idx]
    test_outer_block = all_blocks[outer_idx]

    train_outer_raw = df_sorted[df_sorted['time_block'].isin(train_outer_blocks)].copy()
    test_outer_raw = df_sorted[df_sorted['time_block'] == test_outer_block].copy()

    # outer anti-leakage feature engineering
    [train_outer_fe, test_outer_fe], outer_high_risk_mcc = add_high_risk_mcc(
        train_df=train_outer_raw,
        apply_dfs=[train_outer_raw, test_outer_raw],
        threshold=0.02
    )

    inner_results = []

    for inner_idx in range(1, len(train_outer_blocks)):
        train_inner_blocks = train_outer_blocks[:inner_idx]
        val_inner_block = train_outer_blocks[inner_idx]

        train_inner_raw = train_outer_raw[
            train_outer_raw['time_block'].isin(train_inner_blocks)
        ].copy()

        val_inner_raw = train_outer_raw[
            train_outer_raw['time_block'] == val_inner_block
        ].copy()

        # inner anti-leakage feature engineering
        [train_inner_fe, val_inner_fe], inner_high_risk_mcc = add_high_risk_mcc(
            train_df=train_inner_raw,
            apply_dfs=[train_inner_raw, val_inner_raw],
            threshold=0.02
        )

        drop_final_cols = ['date', 'year', 'time_block']

        train_inner_final = train_inner_fe.drop(
            columns=[c for c in drop_final_cols if c in train_inner_fe.columns]
        ).copy()

        val_inner_final = val_inner_fe.drop(
            columns=[c for c in drop_final_cols if c in val_inner_fe.columns]
        ).copy()

        inner_results.append({
            'inner_fold': inner_idx,
            'train_blocks': train_inner_blocks,
            'val_block': val_inner_block,
            'train_inner': train_inner_final,
            'val_inner': val_inner_final,
            'high_risk_mcc_list': list(inner_high_risk_mcc)
        })

    drop_final_cols = ['date', 'year', 'time_block']

    train_outer_final = train_outer_fe.drop(
        columns=[c for c in drop_final_cols if c in train_outer_fe.columns]
    ).copy()

    test_outer_final = test_outer_fe.drop(
        columns=[c for c in drop_final_cols if c in test_outer_fe.columns]
    ).copy()

    nested_cv_data.append({
        'outer_fold': outer_idx - 1,
        'train_outer_blocks': train_outer_blocks,
        'test_outer_block': test_outer_block,
        'train_outer': train_outer_final,
        'test_outer': test_outer_final,
        'inner_results': inner_results,
        'high_risk_mcc_list_outer': list(outer_high_risk_mcc)
    })


# ===========================================================
# 4. 主迴圈：Nested CV + Feature Groups
# ===========================================================
full_logit_results_nested = {}

for fold_info in nested_cv_data:
    outer_fold = fold_info['outer_fold']
    train_outer = fold_info['train_outer'].copy()
    test_outer = fold_info['test_outer'].copy()
    test_outer_block = fold_info['test_outer_block']

    print(f"\n{'='*60}")
    print(f"🚀 Running Full Logit for OUTER FOLD {outer_fold}")
    print(f"Train outer blocks: {fold_info['train_outer_blocks']}")
    print(f"Test outer block : {test_outer_block}")
    print(f"{'='*60}")

    full_logit_results_nested[outer_fold] = {
        "test_outer_block": test_outer_block,
        "train_outer_blocks": fold_info['train_outer_blocks'],
        "groups": {}
    }

    for group_name, feature_list in feature_groups.items():
        valid_features = [f for f in feature_list if f in train_outer.columns]

        if not valid_features:
            print(f"   ⚠️ Group: {group_name} - No valid features found. Skipping.")
            continue

        print(f"   🔹 Group: {group_name} ({len(valid_features)} features)...", end=" ")

        # =========================
        # 先做 inner CV（只做驗證，不調參也至少記錄）
        # =========================
        inner_fold_metrics = []

        for inner_info in fold_info['inner_results']:
            train_inner = inner_info['train_inner'].copy()
            val_inner = inner_info['val_inner'].copy()

            inner_valid_features = [f for f in valid_features if f in train_inner.columns]

            if not inner_valid_features:
                continue

            inner_res = fit_full_logit(
                train_df=train_inner,
                test_df=val_inner,
                feature_cols=inner_valid_features,
                dep_var="is_fraud"
            )

            if inner_res["status"] == "success":
                inner_metrics = inner_res["metrics"]
                inner_fold_metrics.append({
                    "inner_fold": inner_info["inner_fold"],
                    "train_blocks": inner_info["train_blocks"],
                    "val_block": inner_info["val_block"],
                    "Train AUC": inner_metrics["Train AUC"],
                    "Val AUC": inner_metrics["Test AUC"],
                    "Train PR-AUC": inner_metrics["Train PR-AUC"],
                    "Val PR-AUC": inner_metrics["Test PR-AUC"]
                })

        # inner 平均表現
        if len(inner_fold_metrics) > 0:
            avg_val_auc = round(np.mean([x["Val AUC"] for x in inner_fold_metrics]), 4)
            avg_val_prauc = round(np.mean([x["Val PR-AUC"] for x in inner_fold_metrics]), 4)
        else:
            avg_val_auc = np.nan
            avg_val_prauc = np.nan

        # =========================
        # outer 最終評估
        # =========================
        outer_res = fit_full_logit(
            train_df=train_outer,
            test_df=test_outer,
            feature_cols=valid_features,
            dep_var="is_fraud"
        )

        if outer_res["status"] == "success":
            metrics = outer_res["metrics"]
            print("✅ Done.")
            print(f"      Train AUC: {metrics['Train AUC']} | Test AUC: {metrics['Test AUC']}")
            print(f"      Train PR : {metrics['Train PR-AUC']} | Test PR : {metrics['Test PR-AUC']}")
            print(f"      Avg Val AUC: {avg_val_auc} | Avg Val PR: {avg_val_prauc}")

            full_logit_results_nested[outer_fold]["groups"][group_name] = {
                "Outer Train AUC": metrics["Train AUC"],
                "Outer Test AUC": metrics["Test AUC"],
                "Outer Train PR-AUC": metrics["Train PR-AUC"],
                "Outer Test PR-AUC": metrics["Test PR-AUC"],
                "Avg Inner Val AUC": avg_val_auc,
                "Avg Inner Val PR-AUC": avg_val_prauc,
                "Inner Fold Details": inner_fold_metrics
            }
        else:
            print(f"❌ Error: {outer_res['message']}")
            full_logit_results_nested[outer_fold]["groups"][group_name] = {
                "error": outer_res["message"]
            }

print("\n🎉 All Nested CV Full Logistic Regression models completed!")

/var/folders/yb/xnfk9z6x34527z3924bcjqcr0000gn/T/ipykernel_27178/3883392805.py:116: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  df_new['HighRiskMCC'] = df_new['mcc_code'].isin(high_risk_mcc).astype('uint8')
/var/folders/yb/xnfk9z6x34527z39


🚀 Running Full Logit for OUTER FOLD 1
Train outer blocks: ['2010-2011', '2012-2013']
Test outer block : 2014-2015
   🔹 Group: X_all (19 features)... ✅ Done.
      Train AUC: 0.9048 | Test AUC: 0.8186
      Train PR : 0.0274 | Test PR : 0.0132
      Avg Val AUC: 0.8812 | Avg Val PR: 0.0265
   🔹 Group: X_rfm (7 features)... ✅ Done.
      Train AUC: 0.7495 | Test AUC: 0.7516
      Train PR : 0.006 | Test PR : 0.0117
      Avg Val AUC: 0.741 | Avg Val PR: 0.0076
   🔹 Group: X_dk (7 features)... ✅ Done.
      Train AUC: 0.9074 | Test AUC: 0.8603
      Train PR : 0.0841 | Test PR : 0.0776
      Avg Val AUC: 0.8378 | Avg Val PR: 0.0402
   🔹 Group: X_all + X_rfm (26 features)... ✅ Done.
      Train AUC: 0.9309 | Test AUC: 0.8625
      Train PR : 0.0497 | Test PR : 0.0357
      Avg Val AUC: 0.9121 | Avg Val PR: 0.0599
   🔹 Group: X_all + X_dk (26 features)... ✅ Done.
      Train AUC: 0.9168 | Test AUC: 0.8862
      Train PR : 0.1102 | Test PR : 0.1006
      Avg Val AUC: 0.8517 | Avg Val PR: 0.

#### (6-1 b) Run Logistic Regression

#### (6-1 c) Logistic result output

6-1 c **跑完Logistic後的結果**會輸出在資料夾名為<font color=blue>“logit_results_wide.csv”</font>的csv檔案當中

In [37]:
import pandas as pd
import matplotlib.pyplot as plt

# ==========================================
# 1. 將 nested 結果轉為 DataFrame
# ==========================================
data_list = []

for outer_fold, fold_info in full_logit_results_nested.items():
    test_outer_block = fold_info["test_outer_block"]

    for group_name, metrics in fold_info["groups"].items():
        if "error" in metrics:
            continue

        row = {
            "Outer Fold": outer_fold,
            "Test Block": test_outer_block,
            "Feature Group": group_name,
            "Avg Inner Val AUC": metrics.get("Avg Inner Val AUC"),
            "Avg Inner Val PR-AUC": metrics.get("Avg Inner Val PR-AUC"),
            "Outer Train AUC": metrics.get("Outer Train AUC"),
            "Outer Test AUC": metrics.get("Outer Test AUC"),
            "Outer Train PR-AUC": metrics.get("Outer Train PR-AUC"),
            "Outer Test PR-AUC": metrics.get("Outer Test PR-AUC")
        }
        data_list.append(row)

df_results_nested = pd.DataFrame(data_list)

cols = [
    "Outer Fold", "Test Block", "Feature Group",
    "Avg Inner Val AUC", "Avg Inner Val PR-AUC",
    "Outer Train AUC", "Outer Test AUC",
    "Outer Train PR-AUC", "Outer Test PR-AUC"
]
df_results_nested = df_results_nested[cols]

print("=== Nested CV 詳細結果表 ===")
print(df_results_nested.to_string(index=False))


# ==========================================
# 2. 製作績效矩陣
# ==========================================
inner_val_auc = df_results_nested.pivot(
    index="Feature Group", columns="Test Block", values="Avg Inner Val AUC"
)

inner_val_prauc = df_results_nested.pivot(
    index="Feature Group", columns="Test Block", values="Avg Inner Val PR-AUC"
)

outer_train_auc = df_results_nested.pivot(
    index="Feature Group", columns="Test Block", values="Outer Train AUC"
)

outer_test_auc = df_results_nested.pivot(
    index="Feature Group", columns="Test Block", values="Outer Test AUC"
)

outer_train_prauc = df_results_nested.pivot(
    index="Feature Group", columns="Test Block", values="Outer Train PR-AUC"
)

outer_test_prauc = df_results_nested.pivot(
    index="Feature Group", columns="Test Block", values="Outer Test PR-AUC"
)

print("\n=== Avg Inner Validation AUC 矩陣 ===")
print(inner_val_auc)

print("\n=== Avg Inner Validation PR-AUC 矩陣 ===")
print(inner_val_prauc)

print("\n=== Outer Train AUC 矩陣 ===")
print(outer_train_auc)

print("\n=== Outer Test AUC 矩陣 ===")
print(outer_test_auc)

print("\n=== Outer Train PR-AUC 矩陣 ===")
print(outer_train_prauc)

print("\n=== Outer Test PR-AUC 矩陣 ===")
print(outer_test_prauc)


# ==========================================
# 3. 製作整合大表
# ==========================================
metrics = [
    "Avg Inner Val AUC",
    "Avg Inner Val PR-AUC",
    "Outer Train AUC",
    "Outer Test AUC",
    "Outer Train PR-AUC",
    "Outer Test PR-AUC"
]

df_pivot = df_results_nested.pivot(
    index="Feature Group",
    columns="Test Block",
    values=metrics
)

df_pivot.columns = df_pivot.columns.swaplevel(0, 1)

unique_blocks = sorted(df_results_nested["Test Block"].unique())
ordered_columns = []

for block in unique_blocks:
    for metric in metrics:
        ordered_columns.append((block, metric))

df_final_nested = df_pivot.reindex(columns=ordered_columns)

print("\n=== Nested CV 整合結果總表 ===")
print(df_final_nested.to_string())

df_final_nested.to_csv("logit_results_nested_wide.csv")
df_results_nested.to_csv("logit_results_nested_long.csv", index=False)

=== Nested CV 詳細結果表 ===
 Outer Fold Test Block        Feature Group  Avg Inner Val AUC  Avg Inner Val PR-AUC  Outer Train AUC  Outer Test AUC  Outer Train PR-AUC  Outer Test PR-AUC
          1  2014-2015                X_all             0.8812                0.0265           0.9048          0.8186              0.0274             0.0132
          1  2014-2015                X_rfm             0.7410                0.0076           0.7495          0.7516              0.0060             0.0117
          1  2014-2015                 X_dk             0.8378                0.0402           0.9074          0.8603              0.0841             0.0776
          1  2014-2015        X_all + X_rfm             0.9121                0.0599           0.9309          0.8625              0.0497             0.0357
          1  2014-2015         X_all + X_dk             0.8517                0.0539           0.9168          0.8862              0.1102             0.1006
          1  2014-2015         X_r

#### (6-1 d) Logistic Regression(使用stepwise features 版本--未修正)

6-1 d 用於跑完stepwise features selection 後，直接讀取選完的features來進行邏輯斯回歸，若要單獨運行此段程式，記得先載好6-1 a的function即可！！
**LR 運行stepwise selection的結果**會輸出在資料夾名為<font color=blue>“LR_step.csv”</font>的csv檔案當中

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score

# ===========================================================
# 0. 輔助檢查函數 (跟之前一樣)
# ===========================================================
def check_data_validity(y_data, stage="Train"):
    if len(y_data) == 0:
        return False, f"{stage} set is empty"
    if y_data.nunique() < 2:
        val = y_data.iloc[0] if len(y_data) > 0 else "None"
        return False, f"{stage} set has only 1 class (value: {val})"
    return True, ""

# ===========================================================
# 1. 增強版 Logistic Regression 訓練函數 (含防呆)
# ===========================================================
def fit_stepwise_logit(train_df, test_df, feature_cols, dep_var="is_fraud"):
    """
    針對 Stepwise 篩選後的變數跑 Logistic Regression
    """
    X_train = train_df[feature_cols]
    y_train = train_df[dep_var]
    X_test  = test_df[feature_cols]
    y_test  = test_df[dep_var]

    # --- 1. 防崩潰檢查 ---
    is_valid_train, msg_train = check_data_validity(y_train, "Train")
    if not is_valid_train:
        return {"status": "skip", "message": msg_train}

    is_valid_test, _ = check_data_validity(y_test, "Test")

    # --- 2. 缺失值處理 ---
    # 雖然 Stepwise 階段可能處理過，但防萬一
    X_train = X_train.fillna(0)
    X_test  = X_test.fillna(0)

    # --- 3. 標準化 (Logistic Regression 必備) ---
    try:
        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train)
        X_test_scaled  = scaler.transform(X_test)
    except Exception as e:
        return {"status": "error", "message": f"Scaling error: {str(e)}"}

    # --- 4. 訓練模型 ---
    model = LogisticRegression(
        penalty="l2",
        solver="lbfgs",
        max_iter=2000,
        n_jobs=-1,
        random_state=42
    )
    
    try:
        model.fit(X_train_scaled, y_train)
        
        # --- 5. 計算指標 ---
        train_prob = model.predict_proba(X_train_scaled)[:, 1]
        train_auc  = roc_auc_score(y_train, train_prob)
        train_pr   = average_precision_score(y_train, train_prob)
        
        test_auc = np.nan
        test_pr  = np.nan
        
        if is_valid_test:
            try:
                test_prob = model.predict_proba(X_test_scaled)[:, 1]
                test_auc  = roc_auc_score(y_test, test_prob)
                test_pr   = average_precision_score(y_test, test_prob)
            except:
                pass # 保持 NaN

        return {
            "status": "success",
            "model": model,
            "metrics": {
                "Train AUC": round(train_auc, 4),
                "Test AUC": round(test_auc, 4) if not np.isnan(test_auc) else np.nan,
                "Train PR-AUC": round(train_pr, 4),
                "Test PR-AUC": round(test_pr, 4) if not np.isnan(test_pr) else np.nan
            }
        }
        
    except Exception as e:
        return {"status": "error", "message": str(e)}

# ===========================================================
# 2. 主迴圈 (Stepwise Storage -> Logistic Regression)
# ===========================================================

# 確保必要變數存在
if 'stepwise_feature_storage' not in locals():
    raise ValueError("⚠️ 請先執行 Stepwise 篩選，取得 stepwise_feature_storage！")

logit_stepwise_results = []

# 這裡我們需要 group names 來遍歷字典
group_names = feature_groups.keys()

for block_id, block_df in df_sorted.groupby('time_block'):
    
    current_year = 2010 + block_id
    print(f"\n{'='*60}")
    print(f"📊 Running Logit (Stepwise Feats) for Block {block_id} (Year: {current_year})")
    print(f"{'='*60}")

    # --- 資料切分 (Split) ---
    block_df = block_df.sort_values('date')
    split_index = int(len(block_df) * 0.8)
    
    train_raw = block_df.iloc[:split_index].copy()
    test_raw  = block_df.iloc[split_index:].copy()

    # --- 特徵工程 (Anti-Leakage) ---
    # 必須做！因為 Stepwise 選出的變數裡可能包含 HighRiskMCC
    try:
        fraud_rate = train_raw.groupby('mcc_code')['is_fraud'].mean()
        high_risk_mcc = fraud_rate[fraud_rate > 0.02].index
        train_raw['HighRiskMCC'] = train_raw['mcc_code'].isin(high_risk_mcc).astype('uint8')
        test_raw['HighRiskMCC']  = test_raw['mcc_code'].isin(high_risk_mcc).astype('uint8')
    except:
        pass

    # --- 讀取該年份 Stepwise 結果 ---
    block_feats_dict = stepwise_feature_storage.get(block_id, {})

    # --- 針對每個 Group 跑模型 ---
    for group_name in group_names:
        
        # ★ 關鍵：從字典取出「該年份、該群組」被 Stepwise 選中的變數
        my_features = block_feats_dict.get(group_name, [])
        
        # 情況 A: Stepwise 沒選出任何變數 (或空集合)
        if not my_features:
            print(f"   ⚠️ Group: {group_name} - Skipped (No Stepwise features)")
            logit_stepwise_results.append({
                "Year": current_year, "Feature Group": group_name,
                "Train AUC": np.nan, "Test AUC": np.nan,
                "Train PR-AUC": np.nan, "Test PR-AUC": np.nan,
                "Num Features": 0
            })
            continue

        print(f"   🔹 Group: {group_name} (Using {len(my_features)} features)...", end=" ")

        # 情況 B: 執行 Logistic Regression
        res = fit_stepwise_logit(
            train_raw, 
            test_raw, 
            feature_cols=my_features, # <--- 使用 Stepwise 選出的變數
            dep_var="is_fraud"
        )

        if res["status"] == "success":
            m = res["metrics"]
            print(f"✅ Test AUC: {m.get('Test AUC', np.nan)}")
            
            logit_stepwise_results.append({
                "Year": current_year,
                "Feature Group": group_name,
                "Train AUC": m.get("Train AUC"),
                "Test AUC": m.get("Test AUC"),
                "Train PR-AUC": m.get("Train PR-AUC"),
                "Test PR-AUC": m.get("Test PR-AUC"),
                "Num Features": len(my_features)
            })
            
        elif res["status"] == "skip":
            print(f"⚠️ Skipped: {res['message']}")
            logit_stepwise_results.append({
                "Year": current_year, "Feature Group": group_name,
                "Train AUC": np.nan, "Test AUC": np.nan,
                "Train PR-AUC": np.nan, "Test PR-AUC": np.nan,
                "Num Features": len(my_features)
            })
            
        else:
            print(f"❌ Error: {res['message']}")

print("\n🎉 All Logistic Regression (Stepwise) models completed!")


# ===========================================================
# 3. 產出報表
# ===========================================================
if logit_stepwise_results:
    df_logit_raw = pd.DataFrame(logit_stepwise_results)
    
    # Pivot 轉置
    target_metrics = ["Train AUC", "Test AUC", "Train PR-AUC", "Test PR-AUC"]
    df_logit_pivot = df_logit_raw.pivot(index="Feature Group", columns="Year", values=target_metrics)
    
    # 調整欄位順序
    df_logit_pivot.columns = df_logit_pivot.columns.swaplevel(0, 1)
    unique_years = sorted(df_logit_raw["Year"].unique())
    ordered_columns = [(y, m) for y in unique_years for m in target_metrics]
    
    df_logit_final = df_logit_pivot.reindex(columns=ordered_columns)
    
    print("\n=== Logistic Regression Performance (Using Stepwise Selected Features) ===")
    print(df_logit_final.to_string())
else:
    print("No results generated.")

df_logit_final.to_csv("LR_step.csv")

## 06-2 XGB

#### (6-2 a) XGB function定義

In [38]:
import pandas as pd
import numpy as np
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score, average_precision_score

# ===========================================================
# 1. 定義單次 XGBoost 訓練函數
# ===========================================================
def run_xgb_single(train_df, test_df, feature_cols, dep_var="is_fraud"):
    """
    針對給定的 Train/Test 和特徵列表訓練 XGBoost
    """
    X_train = train_df[feature_cols].copy()
    y_train = train_df[dep_var].copy()

    X_test = test_df[feature_cols].copy()
    y_test = test_df[dep_var].copy()

    # XGBoost 保守處理：避免殘留缺失值
    X_train = X_train.fillna(0)
    X_test = X_test.fillna(0)

    model = XGBClassifier(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        eval_metric="auc",
        random_state=42,
        tree_method="hist",
        n_jobs=-1,
        early_stopping_rounds=50
    )

    try:
        model.fit(
            X_train, y_train,
            eval_set=[(X_test, y_test)],
            verbose=False
        )

        train_pred = model.predict_proba(X_train)[:, 1]
        test_pred = model.predict_proba(X_test)[:, 1]

        train_roc = roc_auc_score(y_train, train_pred)
        test_roc = roc_auc_score(y_test, test_pred)
        train_pr = average_precision_score(y_train, train_pred)
        test_pr = average_precision_score(y_test, test_pred)

        best_iter = model.best_iteration if hasattr(model, "best_iteration") and model.best_iteration is not None else 300

        return {
            "status": "success",
            "model": model,
            "metrics": {
                "Train AUC": round(train_roc, 4),
                "Test AUC": round(test_roc, 4),
                "Train PR-AUC": round(train_pr, 4),
                "Test PR-AUC": round(test_pr, 4),
                "Best Iteration": int(best_iter)
            }
        }

    except Exception as e:
        return {"status": "error", "message": str(e)}


# ===========================================================
# 2. 準備 Feature Groups
# ===========================================================
identifier = ['transaction_id', 'client_id_x', 'card_id', 'card_number', 'cvv']
exclude_cols = set(identifier) | {'is_fraud', 'date', 'year', 'time_block'}

def clean_cols(cols):
    # 去掉 exclude_cols，並保持原順序去重複
    seen = set()
    cleaned = []
    for c in cols:
        if c not in exclude_cols and c not in seen:
            cleaned.append(c)
            seen.add(c)
    return cleaned

feature_groups = {
    "X_all": clean_cols(all_cols),
    "X_rfm": clean_cols(rfm_cols),
    "X_dk": clean_cols(dk_cols),
    "X_all + X_rfm": clean_cols(all_cols + rfm_cols),
    "X_all + X_dk": clean_cols(all_cols + dk_cols),
    "X_rfm + X_dk": clean_cols(rfm_cols + dk_cols),
    "X_all + X_rfm + X_dk": clean_cols(all_cols + rfm_cols + dk_cols)
}


# ===========================================================
# 3. 若 nested_cv_data 尚未建立，先建立
# ===========================================================
def add_high_risk_mcc(train_df, apply_dfs, threshold=0.02):
    """
    train_df: 用來計算 fraud rate 的 training dataframe
    apply_dfs: list of dataframes，要套用 HighRiskMCC 的資料
    threshold: fraud rate 門檻
    """
    if 'mcc_code' not in train_df.columns:
        raise ValueError("缺少 'mcc_code' 欄位")
    if 'is_fraud' not in train_df.columns:
        raise ValueError("缺少 'is_fraud' 欄位")

    fraud_rate = train_df.groupby('mcc_code')['is_fraud'].mean()
    high_risk_mcc = fraud_rate[fraud_rate > threshold].index

    output_dfs = []
    for df in apply_dfs:
        df_new = df.copy()
        df_new['HighRiskMCC'] = df_new['mcc_code'].isin(high_risk_mcc).astype('uint8')
        output_dfs.append(df_new)

    return output_dfs, high_risk_mcc


if 'nested_cv_data' not in locals():
    if 'df_sorted' not in locals():
        raise ValueError("df_sorted is not defined. Please ensure your DataFrame is prepared.")

    all_blocks = sorted(df_sorted['time_block'].unique())
    nested_cv_data = []

    for outer_idx in range(2, len(all_blocks)):
        train_outer_blocks = all_blocks[:outer_idx]
        test_outer_block = all_blocks[outer_idx]

        train_outer_raw = df_sorted[df_sorted['time_block'].isin(train_outer_blocks)].copy()
        test_outer_raw = df_sorted[df_sorted['time_block'] == test_outer_block].copy()

        # outer anti-leakage feature engineering
        [train_outer_fe, test_outer_fe], outer_high_risk_mcc = add_high_risk_mcc(
            train_df=train_outer_raw,
            apply_dfs=[train_outer_raw, test_outer_raw],
            threshold=0.02
        )

        inner_results = []

        for inner_idx in range(1, len(train_outer_blocks)):
            train_inner_blocks = train_outer_blocks[:inner_idx]
            val_inner_block = train_outer_blocks[inner_idx]

            train_inner_raw = train_outer_raw[
                train_outer_raw['time_block'].isin(train_inner_blocks)
            ].copy()

            val_inner_raw = train_outer_raw[
                train_outer_raw['time_block'] == val_inner_block
            ].copy()

            [train_inner_fe, val_inner_fe], inner_high_risk_mcc = add_high_risk_mcc(
                train_df=train_inner_raw,
                apply_dfs=[train_inner_raw, val_inner_raw],
                threshold=0.02
            )

            drop_final_cols = ['date', 'year', 'time_block']

            train_inner_final = train_inner_fe.drop(
                columns=[c for c in drop_final_cols if c in train_inner_fe.columns]
            ).copy()

            val_inner_final = val_inner_fe.drop(
                columns=[c for c in drop_final_cols if c in val_inner_fe.columns]
            ).copy()

            inner_results.append({
                'inner_fold': inner_idx,
                'train_blocks': train_inner_blocks,
                'val_block': val_inner_block,
                'train_inner': train_inner_final,
                'val_inner': val_inner_final,
                'high_risk_mcc_list': list(inner_high_risk_mcc)
            })

        drop_final_cols = ['date', 'year', 'time_block']

        train_outer_final = train_outer_fe.drop(
            columns=[c for c in drop_final_cols if c in train_outer_fe.columns]
        ).copy()

        test_outer_final = test_outer_fe.drop(
            columns=[c for c in drop_final_cols if c in test_outer_fe.columns]
        ).copy()

        nested_cv_data.append({
            'outer_fold': outer_idx - 1,
            'train_outer_blocks': train_outer_blocks,
            'test_outer_block': test_outer_block,
            'train_outer': train_outer_final,
            'test_outer': test_outer_final,
            'inner_results': inner_results,
            'high_risk_mcc_list_outer': list(outer_high_risk_mcc)
        })


# ===========================================================
# 4. 主迴圈：Nested CV + Feature Groups
# ===========================================================
xgb_results_nested = {}

for fold_info in nested_cv_data:
    outer_fold = fold_info['outer_fold']
    train_outer = fold_info['train_outer'].copy()
    test_outer = fold_info['test_outer'].copy()
    test_outer_block = fold_info['test_outer_block']

    print(f"\n{'='*60}")
    print(f"🚀 Running XGBoost for OUTER FOLD {outer_fold}")
    print(f"Train outer blocks: {fold_info['train_outer_blocks']}")
    print(f"Test outer block : {test_outer_block}")
    print(f"{'='*60}")

    xgb_results_nested[outer_fold] = {
        "test_outer_block": test_outer_block,
        "train_outer_blocks": fold_info['train_outer_blocks'],
        "groups": {}
    }

    for group_name, feature_list in feature_groups.items():
        valid_features = [f for f in feature_list if f in train_outer.columns]

        if not valid_features:
            print(f"   ⚠️ Group: {group_name} - No valid features found. Skipping.")
            continue

        print(f"   🔹 Group: {group_name} ({len(valid_features)} features)...", end=" ")

        # =========================
        # 先做 inner CV
        # =========================
        inner_fold_metrics = []

        for inner_info in fold_info['inner_results']:
            train_inner = inner_info['train_inner'].copy()
            val_inner = inner_info['val_inner'].copy()

            inner_valid_features = [f for f in valid_features if f in train_inner.columns]

            if not inner_valid_features:
                continue

            inner_res = run_xgb_single(
                train_df=train_inner,
                test_df=val_inner,
                feature_cols=inner_valid_features,
                dep_var="is_fraud"
            )

            if inner_res["status"] == "success":
                inner_metrics = inner_res["metrics"]
                inner_fold_metrics.append({
                    "inner_fold": inner_info["inner_fold"],
                    "train_blocks": inner_info["train_blocks"],
                    "val_block": inner_info["val_block"],
                    "Train AUC": inner_metrics["Train AUC"],
                    "Val AUC": inner_metrics["Test AUC"],
                    "Train PR-AUC": inner_metrics["Train PR-AUC"],
                    "Val PR-AUC": inner_metrics["Test PR-AUC"],
                    "Best Iteration": inner_metrics["Best Iteration"]
                })

        if len(inner_fold_metrics) > 0:
            avg_val_auc = round(np.mean([x["Val AUC"] for x in inner_fold_metrics]), 4)
            avg_val_prauc = round(np.mean([x["Val PR-AUC"] for x in inner_fold_metrics]), 4)
            avg_best_iter = int(round(np.mean([x["Best Iteration"] for x in inner_fold_metrics])))
        else:
            avg_val_auc = np.nan
            avg_val_prauc = np.nan
            avg_best_iter = np.nan

        # =========================
        # outer 最終評估
        # =========================
        outer_res = run_xgb_single(
            train_df=train_outer,
            test_df=test_outer,
            feature_cols=valid_features,
            dep_var="is_fraud"
        )

        if outer_res["status"] == "success":
            metrics = outer_res["metrics"]
            print(f"✅ Done (Outer Best Iter: {metrics['Best Iteration']})")
            print(f"      Train AUC: {metrics['Train AUC']} | Test AUC: {metrics['Test AUC']}")
            print(f"      Train PR : {metrics['Train PR-AUC']} | Test PR : {metrics['Test PR-AUC']}")
            print(f"      Avg Val AUC: {avg_val_auc} | Avg Val PR: {avg_val_prauc}")

            xgb_results_nested[outer_fold]["groups"][group_name] = {
                "Outer Train AUC": metrics["Train AUC"],
                "Outer Test AUC": metrics["Test AUC"],
                "Outer Train PR-AUC": metrics["Train PR-AUC"],
                "Outer Test PR-AUC": metrics["Test PR-AUC"],
                "Outer Best Iteration": metrics["Best Iteration"],
                "Avg Inner Val AUC": avg_val_auc,
                "Avg Inner Val PR-AUC": avg_val_prauc,
                "Avg Inner Best Iteration": avg_best_iter,
                "Inner Fold Details": inner_fold_metrics
            }
        else:
            print(f"❌ Error: {outer_res['message']}")
            xgb_results_nested[outer_fold]["groups"][group_name] = {
                "error": outer_res["message"]
            }

print("\n🎉 All Nested CV XGBoost models completed!")


🚀 Running XGBoost for OUTER FOLD 1
Train outer blocks: ['2010-2011', '2012-2013']
Test outer block : 2014-2015
   🔹 Group: X_all (19 features)... ✅ Done (Outer Best Iter: 32)
      Train AUC: 0.9457 | Test AUC: 0.8218
      Train PR : 0.1537 | Test PR : 0.0484
      Avg Val AUC: 0.9032 | Avg Val PR: 0.0735
   🔹 Group: X_rfm (7 features)... ✅ Done (Outer Best Iter: 121)
      Train AUC: 0.8902 | Test AUC: 0.8557
      Train PR : 0.0743 | Test PR : 0.0373
      Avg Val AUC: 0.8217 | Avg Val PR: 0.0194
   🔹 Group: X_dk (7 features)... ✅ Done (Outer Best Iter: 59)
      Train AUC: 0.9394 | Test AUC: 0.9199
      Train PR : 0.0905 | Test PR : 0.078
      Avg Val AUC: 0.9015 | Avg Val PR: 0.0418
   🔹 Group: X_all + X_rfm (26 features)... ✅ Done (Outer Best Iter: 77)
      Train AUC: 0.9701 | Test AUC: 0.8828
      Train PR : 0.3048 | Test PR : 0.0581
      Avg Val AUC: 0.932 | Avg Val PR: 0.1228
   🔹 Group: X_all + X_dk (26 features)... ✅ Done (Outer Best Iter: 63)
      Train AUC: 0.9727 |

#### (6-2 b) Run XGB

#### (6-2 c) XGB result output -

6-2 c **跑完XGB後的結果**會輸出在資料夾名為<font color=blue>“XGB_results_wide.csv”</font>的csv檔案當中

In [39]:
import pandas as pd

# ==========================================
# 1. 將 nested 結果轉為 DataFrame
# ==========================================
xgb_data_list = []

for outer_fold, fold_info in xgb_results_nested.items():
    test_outer_block = fold_info["test_outer_block"]

    for group_name, metrics in fold_info["groups"].items():
        if "error" in metrics:
            continue

        row = {
            "Outer Fold": outer_fold,
            "Test Block": test_outer_block,
            "Feature Group": group_name,
            "Avg Inner Val AUC": metrics.get("Avg Inner Val AUC"),
            "Avg Inner Val PR-AUC": metrics.get("Avg Inner Val PR-AUC"),
            "Avg Inner Best Iteration": metrics.get("Avg Inner Best Iteration"),
            "Outer Train AUC": metrics.get("Outer Train AUC"),
            "Outer Test AUC": metrics.get("Outer Test AUC"),
            "Outer Train PR-AUC": metrics.get("Outer Train PR-AUC"),
            "Outer Test PR-AUC": metrics.get("Outer Test PR-AUC"),
            "Outer Best Iteration": metrics.get("Outer Best Iteration")
        }
        xgb_data_list.append(row)

df_xgb_nested = pd.DataFrame(xgb_data_list)

cols = [
    "Outer Fold", "Test Block", "Feature Group",
    "Avg Inner Val AUC", "Avg Inner Val PR-AUC", "Avg Inner Best Iteration",
    "Outer Train AUC", "Outer Test AUC",
    "Outer Train PR-AUC", "Outer Test PR-AUC",
    "Outer Best Iteration"
]
df_xgb_nested = df_xgb_nested[cols]

print("=== Nested CV XGBoost 詳細結果表 ===")
print(df_xgb_nested.to_string(index=False))


# ==========================================
# 2. 製作績效矩陣
# ==========================================
inner_val_auc = df_xgb_nested.pivot(
    index="Feature Group", columns="Test Block", values="Avg Inner Val AUC"
)

inner_val_prauc = df_xgb_nested.pivot(
    index="Feature Group", columns="Test Block", values="Avg Inner Val PR-AUC"
)

outer_train_auc = df_xgb_nested.pivot(
    index="Feature Group", columns="Test Block", values="Outer Train AUC"
)

outer_test_auc = df_xgb_nested.pivot(
    index="Feature Group", columns="Test Block", values="Outer Test AUC"
)

outer_train_prauc = df_xgb_nested.pivot(
    index="Feature Group", columns="Test Block", values="Outer Train PR-AUC"
)

outer_test_prauc = df_xgb_nested.pivot(
    index="Feature Group", columns="Test Block", values="Outer Test PR-AUC"
)

outer_best_iter = df_xgb_nested.pivot(
    index="Feature Group", columns="Test Block", values="Outer Best Iteration"
)

print("\n=== Avg Inner Validation AUC 矩陣 ===")
print(inner_val_auc)

print("\n=== Avg Inner Validation PR-AUC 矩陣 ===")
print(inner_val_prauc)

print("\n=== Outer Train AUC 矩陣 ===")
print(outer_train_auc)

print("\n=== Outer Test AUC 矩陣 ===")
print(outer_test_auc)

print("\n=== Outer Train PR-AUC 矩陣 ===")
print(outer_train_prauc)

print("\n=== Outer Test PR-AUC 矩陣 ===")
print(outer_test_prauc)

print("\n=== Outer Best Iteration 矩陣 ===")
print(outer_best_iter)


# ==========================================
# 3. 製作整合大表
# ==========================================
target_metrics = [
    "Avg Inner Val AUC",
    "Avg Inner Val PR-AUC",
    "Avg Inner Best Iteration",
    "Outer Train AUC",
    "Outer Test AUC",
    "Outer Train PR-AUC",
    "Outer Test PR-AUC",
    "Outer Best Iteration"
]

df_xgb_pivot = df_xgb_nested.pivot(
    index="Feature Group",
    columns="Test Block",
    values=target_metrics
)

df_xgb_pivot.columns = df_xgb_pivot.columns.swaplevel(0, 1)

unique_blocks = sorted(df_xgb_nested["Test Block"].unique())
ordered_columns = []

for block in unique_blocks:
    for metric in target_metrics:
        ordered_columns.append((block, metric))

df_xgb_final = df_xgb_pivot.reindex(columns=ordered_columns)

print("\n=== XGBoost Nested CV 整合結果總表 ===")
print(df_xgb_final.to_string())

df_xgb_final.to_csv("XGB_results_nested_wide.csv")
df_xgb_nested.to_csv("XGB_results_nested_long.csv", index=False)

=== Nested CV XGBoost 詳細結果表 ===
 Outer Fold Test Block        Feature Group  Avg Inner Val AUC  Avg Inner Val PR-AUC  Avg Inner Best Iteration  Outer Train AUC  Outer Test AUC  Outer Train PR-AUC  Outer Test PR-AUC  Outer Best Iteration
          1  2014-2015                X_all             0.9032                0.0735                        29           0.9457          0.8218              0.1537             0.0484                    32
          1  2014-2015                X_rfm             0.8217                0.0194                       145           0.8902          0.8557              0.0743             0.0373                   121
          1  2014-2015                 X_dk             0.9015                0.0418                        37           0.9394          0.9199              0.0905             0.0780                    59
          1  2014-2015        X_all + X_rfm             0.9320                0.1228                       148           0.9701          0.8828     

#### (6-2 d) XGB 使用stepwise selction選取變數重跑

6-2 d 用於跑完stepwise features selection 後，直接讀取選完的features來進行XGB，若要單獨運行此段程式，記得先載好6-2 a的function即可！！
**XGB 運行stepwise selection的結果**會輸出在資料夾名為<font color=blue>“xgb_stepwise.csv”</font>的csv檔案當中

#### (6-2 e) Optional--用來檢查 features importance而已（我在寫code過程中避免資料洩漏的檢查）

In [ ]:
# ... (前面的定義都不用變) ...

for block_id, block_df in df_sorted.groupby('time_block'):
    
    # ... (中間的 split 和 feature group 迴圈都不用變) ...
    # 這裡省略中間代碼，請直接找到儲存結果的那一行
    
        # 執行 XGBoost
        res = run_xgb_single(
            train_raw, 
            test_raw, 
            feature_cols=valid_features, 
            dep_var="is_fraud"
        )

        if res["status"] == "success":
            metrics = res["metrics"]
            print(f"✅ Done (Best Iter: {metrics['Best Iteration']})")
            
            # =========== ⚠️ 修改這裡！ ===========
            # 原本是: xgb_results[block_id][group_name] = metrics
            # 改成下面這樣，把 Model 也存進去：
            xgb_results[block_id][group_name] = {
                "model": res["model"],  # <--- 關鍵！把模型存下來
                "metrics": metrics
            }
            # ====================================
            
        else:
            print(f"❌ Error: {res['message']}")
            xgb_results[block_id][group_name] = {"error": res['message']}

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# 1. 自動抓取最後一個 Time Block 的 ID (避免 Key Error)
last_block_id = max(xgb_results.keys())
print(f"正在分析 Time Block {last_block_id} 的特徵重要性...")

# 2. 取出模型
# 確保您使用的 group_name 跟訓練時一致 (例如 'X_all + X_rfm + X_dk')
target_group = 'X_all + X_rfm + X_dk' 

if target_group in xgb_results[last_block_id]:
    model = xgb_results[last_block_id][target_group]['model']
    
    # 3. 取得特徵重要性 (Gain)
    # Gain 代表該特徵在樹的分裂中帶來了多少資訊增益（最準確的指標）
    importance = model.get_booster().get_score(importance_type='gain')
    
    # 轉成 DataFrame
    fi_df = pd.DataFrame(list(importance.items()), columns=['Feature', 'Importance'])
    
    # 排序並取前 20 名
    fi_df = fi_df.sort_values(by='Importance', ascending=False).head(20)
    
    print(fi_df)

    # 4. 畫圖
    plt.figure(figsize=(12, 8))
    plt.barh(fi_df['Feature'][::-1], fi_df['Importance'][::-1], color='#1f77b4')
    plt.xlabel('Average Gain (Feature Importance)')
    plt.title(f'Top 20 Features Driving the Model (Block {last_block_id})')
    plt.grid(axis='x', linestyle='--', alpha=0.7)
    plt.tight_layout()
    plt.show()

else:
    print(f"找不到 {target_group} 的結果，請檢查您的 feature_groups 名稱")

In [ ]:
# 檢查 merchant_others 裡面的狀況
print("=== Merchant Others 分析 ===")
mask = df_sorted['merchant_others'] == 1
total = mask.sum()
frauds = df_sorted.loc[mask, 'is_fraud'].sum()

print(f"交易總筆數: {total}")
print(f"詐欺筆數: {frauds}")
print(f"詐欺率 (Fraud Rate): {frauds/total:.2%}")
print("-" * 30)
print("全體平均詐欺率:", df_sorted['is_fraud'].mean())

In [ ]:
# ===========================================================
# 4. 自動化執行：Stepwise Features -> XGBoost
# ===========================================================

# 用來存最終 XGBoost 的結果
xgb_stepwise_results = []

# 確保 df_sorted 存在
if 'df_sorted' not in locals():
    raise ValueError("⚠️ Error: 'df_sorted' 未定義")

for block_id, block_df in df_sorted.groupby('time_block'):
    
    current_year = 2010 + block_id
    print(f"\n{'='*60}")
    print(f"🚀 Running XGBoost (w/ Stepwise Feats) for Block {block_id} (Year: {current_year})")
    print(f"{'='*60}")

    # --- 1. 資料切分 (必須與 Stepwise 階段完全一致) ---
    block_df = block_df.sort_values('date')
    split_index = int(len(block_df) * 0.8)
    
    train_raw = block_df.iloc[:split_index].copy()
    test_raw  = block_df.iloc[split_index:].copy()

    # --- 2. 特徵工程 (Anti-Leakage) ---
    try:
        fraud_rate = train_raw.groupby('mcc_code')['is_fraud'].mean()
        high_risk_mcc = fraud_rate[fraud_rate > 0.02].index
        train_raw['HighRiskMCC'] = train_raw['mcc_code'].isin(high_risk_mcc).astype('uint8')
        test_raw['HighRiskMCC']  = test_raw['mcc_code'].isin(high_risk_mcc).astype('uint8')
    except:
        pass # 若資料為空，後面會被檢查擋下

    # --- 3. 取出該年份 Stepwise 選出的特徵字典 ---
    # 如果該年份完全沒跑 Stepwise (例如報錯)，給一個空字典
    block_feats_dict = stepwise_feature_storage.get(block_id, {})

    # --- 4. 針對每個 Group 跑 XGBoost ---
    for group_name in feature_groups.keys():
        
        # 從字典中取出「該年份、該群組」篩選後的變數列表
        my_features = block_feats_dict.get(group_name, [])
        
        # 檢查 1: 是否有變數被選出
        if not my_features:
            print(f"   ⚠️ Group: {group_name} - Skipped (No features selected by Stepwise)")
            # 存入 NaN 以便報表對齊
            xgb_stepwise_results.append({
                "Year": current_year,
                "Feature Group": group_name,
                "Train AUC": np.nan, "Test AUC": np.nan,
                "Train PR-AUC": np.nan, "Test PR-AUC": np.nan,
                "Num Features": 0
            })
            continue

        print(f"   🔥 Group: {group_name} (Using {len(my_features)} features)...", end=" ")

        # 檢查 2: 執行 XGBoost
        # 注意：這裡傳入的是 my_features (Stepwise 篩選後的結果)
        res = run_xgb_single(
            train_raw, 
            test_raw, 
            feature_cols=my_features, # <--- 關鍵在這裡
            dep_var="is_fraud"
        )

        if res["status"] == "success":
            m = res["metrics"]
            print(f"✅ Test AUC: {m.get('Test AUC', np.nan)}")
            
            xgb_stepwise_results.append({
                "Year": current_year,
                "Feature Group": group_name,
                "Train AUC": m.get("Train AUC"),
                "Test AUC": m.get("Test AUC"),
                "Train PR-AUC": m.get("Train PR-AUC"),
                "Test PR-AUC": m.get("Test PR-AUC"),
                "Num Features": len(my_features)
            })
            
        elif res["status"] == "skip":
            print(f"⚠️ Skipped: {res['message']}")
            xgb_stepwise_results.append({
                "Year": current_year, "Feature Group": group_name,
                "Train AUC": np.nan, "Test AUC": np.nan,
                "Train PR-AUC": np.nan, "Test PR-AUC": np.nan,
                "Num Features": len(my_features)
            })
            
        else:
            print(f"❌ Error: {res['message']}")
            # 視情況決定是否紀錄 Error

print("\n🎉 All XGBoost models completed!")

# ===========================================================
# 5. 產出 XGBoost (Stepwise Features) 最終報表
# ===========================================================
if xgb_stepwise_results:
    df_xgb_raw = pd.DataFrame(xgb_stepwise_results)
    
    # 設定指標
    target_metrics = ["Train AUC", "Test AUC", "Train PR-AUC", "Test PR-AUC"]
    
    # 轉置表格 (Pivot)
    df_xgb_pivot = df_xgb_raw.pivot(index="Feature Group", columns="Year", values=target_metrics)
    
    # 調整欄位層級 (Year 在上)
    df_xgb_pivot.columns = df_xgb_pivot.columns.swaplevel(0, 1)
    
    # 排序欄位
    unique_years = sorted(df_xgb_raw["Year"].unique())
    ordered_columns = [(y, m) for y in unique_years for m in target_metrics]
    df_xgb_final = df_xgb_pivot.reindex(columns=ordered_columns)
    
    print("\n=== XGBoost Performance (Using Stepwise Selected Features) ===")
    print(df_xgb_final.to_string())
else:
    print("No results to display.")

df_xgb_final.to_csv("xgb_stepwise.csv") 

## 06-3 Light GBM

### (6-3 a) LGBM function定義

In [40]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.metrics import roc_auc_score, average_precision_score

# ===========================================================
# 1. 輔助檢查函數
# ===========================================================
def check_data_validity(y_data, stage="Train"):
    if len(y_data) == 0:
        return False, f"{stage} set is empty"
    if y_data.nunique() < 2:
        val = y_data.iloc[0] if len(y_data) > 0 else "None"
        return False, f"{stage} set has only 1 class (value: {val})"
    return True, ""


# ===========================================================
# 2. LightGBM 單次訓練函數
# ===========================================================
def run_lgbm_single(train_df, test_df, feature_cols, dep_var="is_fraud"):
    """
    針對給定的 Train/Test 和特徵列表訓練 LightGBM
    並處理邊界情況 (Edge Cases)
    """
    X_train = train_df[feature_cols].copy()
    y_train = train_df[dep_var].copy()
    X_test  = test_df[feature_cols].copy()
    y_test  = test_df[dep_var].copy()

    # 保守處理缺失值
    X_train = X_train.fillna(0)
    X_test = X_test.fillna(0)

    # --- 1. 防崩潰檢查 ---
    is_valid_train, msg_train = check_data_validity(y_train, "Train")
    if not is_valid_train:
        return {"status": "skip", "message": msg_train}

    is_valid_test, _ = check_data_validity(y_test, "Test")

    # --- 2. 處理類別不平衡 ---
    pos = y_train.sum()
    neg = len(y_train) - pos
    spw = neg / pos if pos > 0 else 1

    # --- 3. 初始化模型 ---
    model = lgb.LGBMClassifier(
        n_estimators=500,
        learning_rate=0.02,
        max_bin=128,
        scale_pos_weight=spw,
        min_split_gain=1.0,
        reg_alpha=0.1,
        reg_lambda=5,
        min_child_samples=20,
        max_depth=4,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        n_jobs=-1,
        verbose=-1
    )

    try:
        # 如果 test 無效，改用 train 當 eval_set 防止 early stopping 報錯
        if is_valid_test:
            eval_set = [(X_test, y_test)]
        else:
            eval_set = [(X_train, y_train)]

        callbacks = [lgb.early_stopping(stopping_rounds=50, verbose=False)]

        model.fit(
            X_train, y_train,
            eval_set=eval_set,
            eval_metric="auc",
            callbacks=callbacks
        )

        # --- 4. 計算 Train 指標 ---
        train_pred = model.predict_proba(X_train)[:, 1]
        train_auc = roc_auc_score(y_train, train_pred)
        train_pr = average_precision_score(y_train, train_pred)

        # --- 5. 計算 Test 指標 ---
        test_auc = np.nan
        test_pr = np.nan

        if is_valid_test:
            try:
                test_pred = model.predict_proba(X_test)[:, 1]
                test_auc = roc_auc_score(y_test, test_pred)
                test_pr = average_precision_score(y_test, test_pred)
            except ValueError:
                pass

        best_iter = model.best_iteration_ if model.best_iteration_ else 500

        return {
            "status": "success",
            "model": model,
            "metrics": {
                "Train AUC": round(train_auc, 4),
                "Test AUC": round(test_auc, 4) if not np.isnan(test_auc) else np.nan,
                "Train PR-AUC": round(train_pr, 4),
                "Test PR-AUC": round(test_pr, 4) if not np.isnan(test_pr) else np.nan,
                "Best Iteration": int(best_iter)
            }
        }

    except Exception as e:
        return {"status": "error", "message": str(e)}


# ===========================================================
# 3. Feature Groups
# ===========================================================
identifier = ['transaction_id', 'client_id_x', 'card_id', 'card_number', 'cvv']
exclude_cols = set(identifier) | {'is_fraud', 'date', 'year', 'time_block'}

def clean_cols(cols):
    seen = set()
    cleaned = []
    for c in cols:
        if c not in exclude_cols and c not in seen:
            cleaned.append(c)
            seen.add(c)
    return cleaned

feature_groups = {
    "X_all": clean_cols(all_cols),
    "X_rfm": clean_cols(rfm_cols),
    "X_dk": clean_cols(dk_cols),
    "X_all + X_rfm": clean_cols(all_cols + rfm_cols),
    "X_all + X_dk": clean_cols(all_cols + dk_cols),
    "X_rfm + X_dk": clean_cols(rfm_cols + dk_cols),
    "X_all + X_rfm + X_dk": clean_cols(all_cols + rfm_cols + dk_cols)
}


# ===========================================================
# 4. 若 nested_cv_data 尚未建立，先建立
# ===========================================================
def add_high_risk_mcc(train_df, apply_dfs, threshold=0.02):
    if 'mcc_code' not in train_df.columns:
        raise ValueError("缺少 'mcc_code' 欄位")
    if 'is_fraud' not in train_df.columns:
        raise ValueError("缺少 'is_fraud' 欄位")

    fraud_rate = train_df.groupby('mcc_code')['is_fraud'].mean()
    high_risk_mcc = fraud_rate[fraud_rate > threshold].index

    output_dfs = []
    for df in apply_dfs:
        df_new = df.copy()
        df_new['HighRiskMCC'] = df_new['mcc_code'].isin(high_risk_mcc).astype('uint8')
        output_dfs.append(df_new)

    return output_dfs, high_risk_mcc


if 'nested_cv_data' not in locals():
    if 'df_sorted' not in locals():
        raise ValueError("df_sorted is not defined. Please ensure your DataFrame is prepared.")

    all_blocks = sorted(df_sorted['time_block'].unique())
    nested_cv_data = []

    for outer_idx in range(2, len(all_blocks)):
        train_outer_blocks = all_blocks[:outer_idx]
        test_outer_block = all_blocks[outer_idx]

        train_outer_raw = df_sorted[df_sorted['time_block'].isin(train_outer_blocks)].copy()
        test_outer_raw = df_sorted[df_sorted['time_block'] == test_outer_block].copy()

        [train_outer_fe, test_outer_fe], outer_high_risk_mcc = add_high_risk_mcc(
            train_df=train_outer_raw,
            apply_dfs=[train_outer_raw, test_outer_raw],
            threshold=0.02
        )

        inner_results = []

        for inner_idx in range(1, len(train_outer_blocks)):
            train_inner_blocks = train_outer_blocks[:inner_idx]
            val_inner_block = train_outer_blocks[inner_idx]

            train_inner_raw = train_outer_raw[
                train_outer_raw['time_block'].isin(train_inner_blocks)
            ].copy()

            val_inner_raw = train_outer_raw[
                train_outer_raw['time_block'] == val_inner_block
            ].copy()

            [train_inner_fe, val_inner_fe], inner_high_risk_mcc = add_high_risk_mcc(
                train_df=train_inner_raw,
                apply_dfs=[train_inner_raw, val_inner_raw],
                threshold=0.02
            )

            drop_final_cols = ['date', 'year', 'time_block']

            train_inner_final = train_inner_fe.drop(
                columns=[c for c in drop_final_cols if c in train_inner_fe.columns]
            ).copy()

            val_inner_final = val_inner_fe.drop(
                columns=[c for c in drop_final_cols if c in val_inner_fe.columns]
            ).copy()

            inner_results.append({
                'inner_fold': inner_idx,
                'train_blocks': train_inner_blocks,
                'val_block': val_inner_block,
                'train_inner': train_inner_final,
                'val_inner': val_inner_final,
                'high_risk_mcc_list': list(inner_high_risk_mcc)
            })

        drop_final_cols = ['date', 'year', 'time_block']

        train_outer_final = train_outer_fe.drop(
            columns=[c for c in drop_final_cols if c in train_outer_fe.columns]
        ).copy()

        test_outer_final = test_outer_fe.drop(
            columns=[c for c in drop_final_cols if c in test_outer_fe.columns]
        ).copy()

        nested_cv_data.append({
            'outer_fold': outer_idx - 1,
            'train_outer_blocks': train_outer_blocks,
            'test_outer_block': test_outer_block,
            'train_outer': train_outer_final,
            'test_outer': test_outer_final,
            'inner_results': inner_results,
            'high_risk_mcc_list_outer': list(outer_high_risk_mcc)
        })


# ===========================================================
# 5. 主迴圈：Nested CV + Feature Groups
# ===========================================================
lgbm_results_nested = {}

for fold_info in nested_cv_data:
    outer_fold = fold_info['outer_fold']
    train_outer = fold_info['train_outer'].copy()
    test_outer = fold_info['test_outer'].copy()
    test_outer_block = fold_info['test_outer_block']

    print(f"\n{'='*60}")
    print(f"🚀 Running LightGBM for OUTER FOLD {outer_fold}")
    print(f"Train outer blocks: {fold_info['train_outer_blocks']}")
    print(f"Test outer block : {test_outer_block}")
    print(f"{'='*60}")

    lgbm_results_nested[outer_fold] = {
        "test_outer_block": test_outer_block,
        "train_outer_blocks": fold_info['train_outer_blocks'],
        "groups": {}
    }

    for group_name, feature_list in feature_groups.items():
        valid_features = [f for f in feature_list if f in train_outer.columns]

        # 強制加入動態特徵 HighRiskMCC
        if 'HighRiskMCC' in train_outer.columns and 'HighRiskMCC' not in valid_features:
            valid_features.append('HighRiskMCC')

        if not valid_features:
            print(f"   ⚠️ Group: {group_name} - No valid features found. Skipping.")
            continue

        print(f"   🔹 Group: {group_name} ({len(valid_features)} features)...", end=" ")

        # =========================
        # inner CV
        # =========================
        inner_fold_metrics = []

        for inner_info in fold_info['inner_results']:
            train_inner = inner_info['train_inner'].copy()
            val_inner = inner_info['val_inner'].copy()

            inner_valid_features = [f for f in valid_features if f in train_inner.columns]

            if 'HighRiskMCC' in train_inner.columns and 'HighRiskMCC' not in inner_valid_features:
                inner_valid_features.append('HighRiskMCC')

            if not inner_valid_features:
                continue

            inner_res = run_lgbm_single(
                train_df=train_inner,
                test_df=val_inner,
                feature_cols=inner_valid_features,
                dep_var="is_fraud"
            )

            if inner_res["status"] == "success":
                inner_metrics = inner_res["metrics"]
                inner_fold_metrics.append({
                    "inner_fold": inner_info["inner_fold"],
                    "train_blocks": inner_info["train_blocks"],
                    "val_block": inner_info["val_block"],
                    "Train AUC": inner_metrics["Train AUC"],
                    "Val AUC": inner_metrics["Test AUC"],
                    "Train PR-AUC": inner_metrics["Train PR-AUC"],
                    "Val PR-AUC": inner_metrics["Test PR-AUC"],
                    "Best Iteration": inner_metrics["Best Iteration"]
                })

            elif inner_res["status"] == "skip":
                inner_fold_metrics.append({
                    "inner_fold": inner_info["inner_fold"],
                    "train_blocks": inner_info["train_blocks"],
                    "val_block": inner_info["val_block"],
                    "Train AUC": np.nan,
                    "Val AUC": np.nan,
                    "Train PR-AUC": np.nan,
                    "Val PR-AUC": np.nan,
                    "Best Iteration": np.nan,
                    "skipped": True,
                    "reason": inner_res["message"]
                })

        valid_inner_auc = [x["Val AUC"] for x in inner_fold_metrics if not pd.isna(x["Val AUC"])]
        valid_inner_pr = [x["Val PR-AUC"] for x in inner_fold_metrics if not pd.isna(x["Val PR-AUC"])]
        valid_inner_iter = [x["Best Iteration"] for x in inner_fold_metrics if not pd.isna(x["Best Iteration"])]

        avg_val_auc = round(np.mean(valid_inner_auc), 4) if len(valid_inner_auc) > 0 else np.nan
        avg_val_prauc = round(np.mean(valid_inner_pr), 4) if len(valid_inner_pr) > 0 else np.nan
        avg_best_iter = int(round(np.mean(valid_inner_iter))) if len(valid_inner_iter) > 0 else np.nan

        # =========================
        # outer 最終評估
        # =========================
        outer_res = run_lgbm_single(
            train_df=train_outer,
            test_df=test_outer,
            feature_cols=valid_features,
            dep_var="is_fraud"
        )

        if outer_res["status"] == "success":
            metrics = outer_res["metrics"]
            print(f"✅ Done (Outer Best Iter: {metrics['Best Iteration']})")
            print(f"      Train AUC: {metrics['Train AUC']} | Test AUC: {metrics['Test AUC']}")
            print(f"      Train PR : {metrics['Train PR-AUC']} | Test PR : {metrics['Test PR-AUC']}")
            print(f"      Avg Val AUC: {avg_val_auc} | Avg Val PR: {avg_val_prauc}")

            lgbm_results_nested[outer_fold]["groups"][group_name] = {
                "Outer Train AUC": metrics["Train AUC"],
                "Outer Test AUC": metrics["Test AUC"],
                "Outer Train PR-AUC": metrics["Train PR-AUC"],
                "Outer Test PR-AUC": metrics["Test PR-AUC"],
                "Outer Best Iteration": metrics["Best Iteration"],
                "Avg Inner Val AUC": avg_val_auc,
                "Avg Inner Val PR-AUC": avg_val_prauc,
                "Avg Inner Best Iteration": avg_best_iter,
                "Inner Fold Details": inner_fold_metrics
            }

        elif outer_res["status"] == "skip":
            print(f"⚠️ Skipped: {outer_res['message']}")
            lgbm_results_nested[outer_fold]["groups"][group_name] = {
                "skipped": True,
                "reason": outer_res["message"]
            }

        else:
            print(f"❌ Error: {outer_res['message']}")
            lgbm_results_nested[outer_fold]["groups"][group_name] = {
                "error": outer_res["message"]
            }

print("\n🎉 All Nested CV LightGBM models completed!")


🚀 Running LightGBM for OUTER FOLD 1
Train outer blocks: ['2010-2011', '2012-2013']
Test outer block : 2014-2015
   🔹 Group: X_all (20 features)... ✅ Done (Outer Best Iter: 4)
      Train AUC: 0.9299 | Test AUC: 0.8278
      Train PR : 0.0587 | Test PR : 0.0078
      Avg Val AUC: 0.8988 | Avg Val PR: 0.0612
   🔹 Group: X_rfm (8 features)... ✅ Done (Outer Best Iter: 500)
      Train AUC: 0.8847 | Test AUC: 0.8598
      Train PR : 0.0626 | Test PR : 0.054
      Avg Val AUC: 0.8205 | Avg Val PR: 0.0233
   🔹 Group: X_dk (7 features)... ✅ Done (Outer Best Iter: 139)
      Train AUC: 0.9394 | Test AUC: 0.9199
      Train PR : 0.09 | Test PR : 0.078
      Avg Val AUC: 0.9025 | Avg Val PR: 0.0418
   🔹 Group: X_all + X_rfm (27 features)... ✅ Done (Outer Best Iter: 498)
      Train AUC: 0.9736 | Test AUC: 0.8852
      Train PR : 0.2372 | Test PR : 0.0769
      Avg Val AUC: 0.9185 | Avg Val PR: 0.0961
   🔹 Group: X_all + X_dk (26 features)... ✅ Done (Outer Best Iter: 221)
      Train AUC: 0.9708 

### (6-3 b) Run LGBM

### (6-3 c) LGBM result output

6-3 c **跑完LGBM後的結果**會輸出在資料夾名為<font color=blue>“lgbm_results_wide.csv”</font>的csv檔案當中

In [41]:
import pandas as pd

# ==========================================
# 1. 將 nested 結果轉為 DataFrame
# ==========================================
lgbm_data_list = []

for outer_fold, fold_info in lgbm_results_nested.items():
    test_outer_block = fold_info["test_outer_block"]

    for group_name, metrics in fold_info["groups"].items():
        if "error" in metrics or "skipped" in metrics:
            continue

        row = {
            "Outer Fold": outer_fold,
            "Test Block": test_outer_block,
            "Feature Group": group_name,
            "Avg Inner Val AUC": metrics.get("Avg Inner Val AUC"),
            "Avg Inner Val PR-AUC": metrics.get("Avg Inner Val PR-AUC"),
            "Avg Inner Best Iteration": metrics.get("Avg Inner Best Iteration"),
            "Outer Train AUC": metrics.get("Outer Train AUC"),
            "Outer Test AUC": metrics.get("Outer Test AUC"),
            "Outer Train PR-AUC": metrics.get("Outer Train PR-AUC"),
            "Outer Test PR-AUC": metrics.get("Outer Test PR-AUC"),
            "Outer Best Iteration": metrics.get("Outer Best Iteration")
        }
        lgbm_data_list.append(row)

df_lgbm_nested = pd.DataFrame(lgbm_data_list)

cols = [
    "Outer Fold", "Test Block", "Feature Group",
    "Avg Inner Val AUC", "Avg Inner Val PR-AUC", "Avg Inner Best Iteration",
    "Outer Train AUC", "Outer Test AUC",
    "Outer Train PR-AUC", "Outer Test PR-AUC",
    "Outer Best Iteration"
]
df_lgbm_nested = df_lgbm_nested[cols]

print("=== Nested CV LightGBM 詳細結果表 ===")
print(df_lgbm_nested.to_string(index=False))


# ==========================================
# 2. 製作績效矩陣
# ==========================================
inner_val_auc = df_lgbm_nested.pivot(
    index="Feature Group", columns="Test Block", values="Avg Inner Val AUC"
)

inner_val_prauc = df_lgbm_nested.pivot(
    index="Feature Group", columns="Test Block", values="Avg Inner Val PR-AUC"
)

outer_train_auc = df_lgbm_nested.pivot(
    index="Feature Group", columns="Test Block", values="Outer Train AUC"
)

outer_test_auc = df_lgbm_nested.pivot(
    index="Feature Group", columns="Test Block", values="Outer Test AUC"
)

outer_train_prauc = df_lgbm_nested.pivot(
    index="Feature Group", columns="Test Block", values="Outer Train PR-AUC"
)

outer_test_prauc = df_lgbm_nested.pivot(
    index="Feature Group", columns="Test Block", values="Outer Test PR-AUC"
)

outer_best_iter = df_lgbm_nested.pivot(
    index="Feature Group", columns="Test Block", values="Outer Best Iteration"
)

print("\n=== Avg Inner Validation AUC 矩陣 ===")
print(inner_val_auc)

print("\n=== Avg Inner Validation PR-AUC 矩陣 ===")
print(inner_val_prauc)

print("\n=== Outer Train AUC 矩陣 ===")
print(outer_train_auc)

print("\n=== Outer Test AUC 矩陣 ===")
print(outer_test_auc)

print("\n=== Outer Train PR-AUC 矩陣 ===")
print(outer_train_prauc)

print("\n=== Outer Test PR-AUC 矩陣 ===")
print(outer_test_prauc)

print("\n=== Outer Best Iteration 矩陣 ===")
print(outer_best_iter)


# ==========================================
# 3. 製作整合大表
# ==========================================
target_metrics = [
    "Avg Inner Val AUC",
    "Avg Inner Val PR-AUC",
    "Avg Inner Best Iteration",
    "Outer Train AUC",
    "Outer Test AUC",
    "Outer Train PR-AUC",
    "Outer Test PR-AUC",
    "Outer Best Iteration"
]

df_lgbm_pivot = df_lgbm_nested.pivot(
    index="Feature Group",
    columns="Test Block",
    values=target_metrics
)

df_lgbm_pivot.columns = df_lgbm_pivot.columns.swaplevel(0, 1)

unique_blocks = sorted(df_lgbm_nested["Test Block"].unique())
ordered_columns = []

for block in unique_blocks:
    for metric in target_metrics:
        ordered_columns.append((block, metric))

df_lgbm_final = df_lgbm_pivot.reindex(columns=ordered_columns)

print("\n=== LightGBM Nested CV 整合結果總表 ===")
print(df_lgbm_final.to_string())

df_lgbm_final.to_csv("lgbm_results_nested_wide.csv")
df_lgbm_nested.to_csv("lgbm_results_nested_long.csv", index=False)

=== Nested CV LightGBM 詳細結果表 ===
 Outer Fold Test Block        Feature Group  Avg Inner Val AUC  Avg Inner Val PR-AUC  Avg Inner Best Iteration  Outer Train AUC  Outer Test AUC  Outer Train PR-AUC  Outer Test PR-AUC  Outer Best Iteration
          1  2014-2015                X_all             0.8988                0.0612                        26           0.9299          0.8278              0.0587             0.0078                     4
          1  2014-2015                X_rfm             0.8205                0.0233                       499           0.8847          0.8598              0.0626             0.0540                   500
          1  2014-2015                 X_dk             0.9025                0.0418                        68           0.9394          0.9199              0.0900             0.0780                   139
          1  2014-2015        X_all + X_rfm             0.9185                0.0961                       474           0.9736          0.8852    

### (6-3 d) LGBM 使用stepwise 版本

6-3 d 用於跑完stepwise features selection 後，直接讀取選完的features來進行LGBM，若要單獨運行此段程式，記得先載好6-3 a的function即可！！
**LGBM 運行stepwise selection的結果**會輸出在資料夾名為<font color=blue>“lgbm_stepwise.csv”</font>的csv檔案當中

In [ ]:
# ===========================================================
# 2. 自動化執行：Stepwise Features -> LightGBM
# ===========================================================

lgbm_stepwise_results = []

# 檢查必要變數
if 'df_sorted' not in locals():
    raise ValueError("⚠️ Error: 'df_sorted' 未定義")
if 'stepwise_feature_storage' not in locals():
    raise ValueError("⚠️ Error: 'stepwise_feature_storage' 未定義，請先執行 Stepwise 步驟。")

# 這裡我們仍需要 feature_groups 的 key (例如 "X_all", "X_rfm") 來跑迴圈
group_names = feature_groups.keys()

for block_id, block_df in df_sorted.groupby('time_block'):
    
    current_year = 2010 + block_id
    print(f"\n{'='*60}")
    print(f"🚀 Running LightGBM (Stepwise Feats) for Block {block_id} (Year: {current_year})")
    print(f"{'='*60}")

    # --- 1. 資料切分 (Split) - 保持一致 ---
    block_df = block_df.sort_values('date')
    split_index = int(len(block_df) * 0.8)
    
    train_raw = block_df.iloc[:split_index].copy()
    test_raw  = block_df.iloc[split_index:].copy()

    # --- 2. 特徵工程 (Anti-Leakage) ---
    try:
        fraud_rate = train_raw.groupby('mcc_code')['is_fraud'].mean()
        high_risk_mcc = fraud_rate[fraud_rate > 0.02].index
        train_raw['HighRiskMCC'] = train_raw['mcc_code'].isin(high_risk_mcc).astype('uint8')
        test_raw['HighRiskMCC']  = test_raw['mcc_code'].isin(high_risk_mcc).astype('uint8')
    except:
        pass # 若資料為空，稍後會被 skip

    # --- 3. 讀取該年份的 Stepwise 變數字典 ---
    # 如果該年份完全沒跑出結果 (例如資料太少報錯)，給空字典
    block_feats_dict = stepwise_feature_storage.get(block_id, {})

    # --- 4. 針對每個 Group 執行 ---
    for group_name in group_names:
        
        # ★ 關鍵：從字典取出 Stepwise 篩選後的變數
        my_features = block_feats_dict.get(group_name, [])
        
        # 情況 A: Stepwise 沒選出任何變數 (或該組資料有問題)
        if not my_features:
            print(f"   ⚠️ Group: {group_name} - Skipped (No features from Stepwise)")
            lgbm_stepwise_results.append({
                "Year": current_year, "Feature Group": group_name,
                "Train AUC": np.nan, "Test AUC": np.nan,
                "Train PR-AUC": np.nan, "Test PR-AUC": np.nan,
                "Num Features": 0
            })
            continue

        print(f"   🔹 Group: {group_name} (Using {len(my_features)} feats)...", end=" ")

        # 情況 B: 執行 LightGBM
        res = run_lgbm_single(
            train_raw, 
            test_raw, 
            feature_cols=my_features, # 使用篩選後的變數
            dep_var="is_fraud"
        )

        if res["status"] == "success":
            m = res["metrics"]
            print(f"✅ Test AUC: {m.get('Test AUC', np.nan)}")
            
            lgbm_stepwise_results.append({
                "Year": current_year,
                "Feature Group": group_name,
                "Train AUC": m.get("Train AUC"),
                "Test AUC": m.get("Test AUC"),
                "Train PR-AUC": m.get("Train PR-AUC"),
                "Test PR-AUC": m.get("Test PR-AUC"),
                "Num Features": len(my_features)
            })
            
        elif res["status"] == "skip":
            print(f"⚠️ Skipped: {res['message']}")
            lgbm_stepwise_results.append({
                "Year": current_year, "Feature Group": group_name,
                "Train AUC": np.nan, "Test AUC": np.nan,
                "Train PR-AUC": np.nan, "Test PR-AUC": np.nan,
                "Num Features": len(my_features)
            })
            
        else:
            print(f"❌ Error: {res['message']}")
            # 視需求決定是否紀錄 Error

print("\n🎉 All LightGBM models (Stepwise) completed!")

# ===========================================================
# 3. 產出報表
# ===========================================================
if lgbm_stepwise_results:
    df_lgbm_raw = pd.DataFrame(lgbm_stepwise_results)
    
    # 轉置
    target_metrics = ["Train AUC", "Test AUC", "Train PR-AUC", "Test PR-AUC"]
    df_lgbm_pivot = df_lgbm_raw.pivot(index="Feature Group", columns="Year", values=target_metrics)
    
    # 調整欄位
    df_lgbm_pivot.columns = df_lgbm_pivot.columns.swaplevel(0, 1)
    unique_years = sorted(df_lgbm_raw["Year"].unique())
    ordered_columns = [(y, m) for y in unique_years for m in target_metrics]
    
    df_lgbm_final = df_lgbm_pivot.reindex(columns=ordered_columns)
    
    print("\n=== LightGBM Performance (Using Stepwise Selected Features) ===")
    print(df_lgbm_final.to_string())
else:
    print("No results generated.")


df_lgbm_final.to_csv("lgbm_stepwise.csv") 

## Optional code--單獨讀取stepwise結果進入code
### (省時間讀stepwise後的結果，用來運行session 6 各d part)

In [ ]:
import pickle
with open('stepwise_feature_storage.pkl', 'rb') as f:
     stepwise_feature_storage = pickle.load(f)
print("✅ 已讀取變數設定")

In [ ]:
import pandas as pd

# 1. 確保它是 DataFrame。如果是字典轉過來的，這步會處理好。
df = pd.DataFrame(stepwise_feature_storage)

# 2. 使用 stack() 將欄位「旋轉」下來
# 這會產生一個多重索引 (Year_Index, Model_Type)
stacked = df.stack()

# 3. 轉回 DataFrame 並重整索引
df_final = stacked.reset_index()

# 4. 重新命名欄位
df_final.columns = ['Year', 'Model_Type', 'Features']

# 5. 將 Year 轉換成你想要的 "year_0" 格式
df_final['Year'] = df_final['Year'].apply(lambda x: f"year_{x}")

# 6. 依照年份排序
df_final = df_final.sort_values(by=['Year', 'Model_Type'])

# --- 輸出結果檢查 ---
for _, row in df_final.iterrows():
    print(f"{row['Year']}  {row['Model_Type']}  {row['Features']}")

# --- 儲存 CSV ---
df_final.to_csv('stepwise_features_final.csv', index=False, encoding='utf-8-sig')
print("\n✅ 最終修正版 CSV 已存檔：stepwise_features_final.csv")